# Correctness tests: reproducing the paper's numerical-check claims

The paper states, in a number of places, that a closed-form expression was
"verified numerically", "checked against direct numerical integration",
"checked against a finite-difference derivative", or similar. This
notebook re-runs every one of those checks independently, using the
modules in this directory, and reports a PASS/FAIL against the threshold
the paper itself states (or, where the paper gives no explicit number,
against machine/quadrature precision).

**Two parts**, in the order the underlying claims appear: Part A checks
the field and self-inductance derivations of the supplementary derivations
(`suppl1`); Part B checks the numbers quoted in the body of the paper (v5),
and the numerical statements of the supplement that support them. Each
check quotes the sentence it tests and names where it appears: "body, Sec.
n" for the paper, "supplementary derivations, ..." for `suppl1`. Checks
marked *supporting* test something the code relies on that the text does
not quote as a number. Derivative checks use
`mpmath` at extended precision rather than plain `numpy` float64: several
closed forms here (`corner_flux`, `self_inductance_correction`) involve
near-cancelling terms at large arguments, and a naive float64 finite
difference can show a spuriously large relative error from that
cancellation alone -- exactly why the paper specifies "high-precision"
finite differences in these places.

**Runtime**: about a minute with `numba`, dominated by the simulation runs
of Part B (B13 repeats the two runs of the paper's Fig. 2, about five
million steps, and B15 the $\varepsilon$ sweep behind Sec. 3.4). `numba` does
not affect any result, but without it Part B takes hours instead of
seconds; see `requirements.txt`.

In [1]:
import sys
sys.path.insert(0, '.')

import numpy as np
import mpmath as mp
from scipy import integrate as sciint, optimize as sciopt

import inductance, field, dynamics, models

mp.mp.dps = 40  # extended precision for derivative/cancellation-sensitive checks

# (x, l, d) triples spanning the reduced-unit default (d=1), a smaller
# circuit, and a case with d != 1, so no check is accidentally tied to d=1.
TEST_PARAMS = [(63.0, 37.0, 1.0), (15.0, 8.0, 1.0), (500.0, 90.0, 2.0)]

PASS, FAIL = [], []
def report(claim, measured, threshold, ok, note=''):
    status = 'PASS' if ok else 'FAIL'
    (PASS if ok else FAIL).append(claim)
    print(f"[{status}] {claim}: measured={measured}  threshold={threshold}  {note}")


## Part A: appendix-level checks (supplementary derivations)

### A1. Corner-field continuity and finiteness

> "All four expressions are finite at the corner point itself and match
> continuously with $B_{ind}^{in,rt}$, $B_{ind}^{in,lt}$, $B_{ind}^{in,up}$
> and $B_{ind}^{in,lo}$ across the boundaries of their respective
> $d\times d$ domains, as verified numerically." (supplementary
> derivations, "Field at the corners")

Each corner region shares a boundary line with two non-corner strips;
check the two field formulas agree exactly along that shared boundary,
and that the corner formulas approach a finite limit as the true corner
point is approached (evaluating exactly at the corner point itself hits a
removable $0/0$).

In [2]:
max_mismatch = 0.0
for (x, l, d) in TEST_PARAMS:
    hh = np.linspace(0.05*d, 0.95*d, 9)
    pp = np.linspace(0.05*d, 0.95*d, 9)
    diffs = [
        np.max(np.abs(field.f_in_lo(d,   hh, x, l, d) - field.f_lo_lt(d,   hh, x, l, d))),
        np.max(np.abs(field.f_in_lo(x-d, hh, x, l, d) - field.f_lo_rt(x-d, hh, x, l, d))),
        np.max(np.abs(field.f_in_up(d,   l-hh, x, l, d) - field.f_up_lt(d,   l-hh, x, l, d))),
        np.max(np.abs(field.f_in_up(x-d, l-hh, x, l, d) - field.f_up_rt(x-d, l-hh, x, l, d))),
        np.max(np.abs(field.f_in_lt(pp, d,   x, l, d) - field.f_lo_lt(pp, d,   x, l, d))),
        np.max(np.abs(field.f_in_lt(pp, l-d, x, l, d) - field.f_up_lt(pp, l-d, x, l, d))),
        np.max(np.abs(field.f_in_rt(x-pp, d,   x, l, d) - field.f_lo_rt(x-pp, d,   x, l, d))),
        np.max(np.abs(field.f_in_rt(x-pp, l-d, x, l, d) - field.f_up_rt(x-pp, l-d, x, l, d))),
    ]
    max_mismatch = max(max_mismatch, max(diffs))
report('A1 corner-strip continuity', f'{max_mismatch:.2e}', '~0 (machine precision)', max_mismatch < 1e-9)

print()
print("Finiteness at the corner point, as eps -> 0 along p=x-eps, h=eps:")
finite_ok = True
for (x, l, d) in [(63.0, 37.0, 1.0), (15.0, 8.0, 1.0)]:
    epss = np.array([1e-2, 1e-4, 1e-6, 1e-8, 1e-10])
    vals = np.array([field.f_lo_rt(x-eps, eps, x, l, d) for eps in epss])
    diffs = np.abs(np.diff(vals))
    shrinking = np.all(np.diff(diffs) < 0)
    finite_ok &= shrinking and np.all(np.isfinite(vals))
    print(f"  x={x}, l={l}: values -> {vals[-1]:.6f}, successive diffs shrink: {shrinking}")
report('A1 corner-point finiteness', 'convergent, finite', 'convergent, finite', finite_ok)


[PASS] A1 corner-strip continuity: measured=1.11e-16  threshold=~0 (machine precision)  

Finiteness at the corner point, as eps -> 0 along p=x-eps, h=eps:
  x=63.0, l=37.0: values -> 0.002494, successive diffs shrink: True
  x=15.0, l=8.0: values -> 0.011273, successive diffs shrink: True
[PASS] A1 corner-point finiteness: measured=convergent, finite  threshold=convergent, finite  


### A2. Total inside-wire + corner flux vs. direct double integration

> "(All of the above, including the corner-splitting convention, has been
> checked numerically against direct double integration of [the four
> corner-flux integrals] and of the updated $A_1$, $A_2$.)"
> (supplementary derivations, "Field at the corners")

Checks that $A_1+B_1+C_1+A_2+B_2+C_2$ (the closed form inside
`inductance.self_inductance_inwire`, minus its `Delta L` term) equals the
direct double integral of the field over all four non-corner strips and
four corners combined.

In [3]:
def total_over_8_regions(fs, x, l, d):
    def ir(f, plo, phi, hlo, hhi):
        return sciint.dblquad(lambda h, p: f(p, h, x, l, d), plo, phi, hlo, hhi)[0]
    v = 0.0
    v += ir(fs['rt'],   x-d, x,   d,   l-d)
    v += ir(fs['lt'],   0,   d,   d,   l-d)
    v += ir(fs['lo'],   d,   x-d, 0,   d)
    v += ir(fs['up'],   d,   x-d, l-d, l)
    v += ir(fs['lort'], x-d, x,   0,   d)
    v += ir(fs['uprt'], x-d, x,   l-d, l)
    v += ir(fs['lolt'], 0,   d,   0,   d)
    v += ir(fs['uplt'], 0,   d,   l-d, l)
    return v

max_reldiff = 0.0
for (x, l, d) in TEST_PARAMS:
    closed = inductance.self_inductance_inwire(1.0, d, l, x) - inductance.self_inductance_correction(1.0, d, l, x)
    direct = total_over_8_regions(field.UNWEIGHTED, x, l, d)
    reldiff = abs(closed - direct) / abs(direct)
    max_reldiff = max(max_reldiff, reldiff)
    print(f"  x={x}, l={l}, d={d}: closed={closed:.8f}  direct={direct:.8f}  reldiff={reldiff:.2e}")
report('A2 inside+corner flux vs. direct double integration', f'{max_reldiff:.2e}', '~0 (quadrature precision)', max_reldiff < 1e-6)


  x=63.0, l=37.0, d=1.0: closed=18.66853924  direct=18.66853924  reldiff=1.09e-13
  x=15.0, l=8.0, d=1.0: closed=5.46155924  direct=5.46155924  reldiff=1.58e-14
  x=500.0, l=90.0, d=2.0: closed=102.22056430  direct=102.22056430  reldiff=4.10e-12
[PASS] A2 inside+corner flux vs. direct double integration: measured=4.10e-12  threshold=~0 (quadrature precision)  


### A3. Derivatives w.r.t. $x$

> "which was likewise verified against a (high-precision) finite-difference
> derivative of [the corner term $\mathcal{D}(x,l)$]."; "All expressions
> above were verified against a numerical (finite-difference) derivative
> of the corresponding functions." (supplementary derivations,
> "Derivatives with respect to $x$")

`inductance.py` does not expose $\partial_xA_1$ etc. as separate
functions (only the assembled `self_inductance_dx`), so this section
differentiates the full closed forms `corner_flux` (the corner term
$\mathcal{D}(x,l)$) and `self_inductance` ($L(x,l)$) themselves. As
demonstrated directly in section A6 below, a naive float64
finite difference of `self_inductance` gives a spuriously large error
here purely from cancellation among its several $O(x^3)$-and-larger
terms, not from any error in the formula -- exactly why the paper
specifies "high-precision" finite differences in this section. This
notebook therefore differentiates a full `mpmath` (extended-precision)
port of every closed form used, not just the small cancellation-prone
pieces.

In [4]:
def mp_fnorm(a, b): return mp.sqrt(a**2 + b**2)
def mp_chi(a, b): return a / mp_fnorm(a, b)
def mp_zeta(a, b):
    r = mp_fnorm(a, b)
    return mp.log((r-a)/(r+a)) / 2
def mp_eta(a, b): return a*mp_zeta(a, b) + b*mp_zeta(b, a)

def mp_corner_flux(x, l, d):
    bracket1 = (mp_fnorm(l,d)**3-l**3 - mp_fnorm(l-d,d)**3+(l-d)**3 + 4*d**3*(mp.sqrt(2)-1)
        + mp_fnorm(x,d)**3-x**3 - mp_fnorm(x-d,d)**3+(x-d)**3)/(3*d**2)
    term_l = d*(mp_zeta(d,l) - mp_zeta(d,l-d)) + mp_fnorm(d,l) - mp_fnorm(d,l-d) - d
    term_x = d*(mp_zeta(d,x) - mp_zeta(d,x-d)) + mp_fnorm(x,d) - mp_fnorm(x-d,d) - d
    mixed_norm = 2*(mp_fnorm(x,l) - mp_fnorm(x,l-d) - mp_fnorm(x-d,l) + mp_fnorm(x-d,l-d))
    mixed_eta = mp_eta(x,l) - mp_eta(x,l-d) - mp_eta(x-d,l) + mp_eta(x-d,l-d)
    return bracket1 + term_l + term_x + mixed_norm + mixed_eta

def mp_phi_selfL(v, d):
    R = mp.sqrt(d**2 + v**2)
    return (v*R*(v**2 - 2*d**2) - v**4 - 6*d**2*v**2 - 2*d**4) / ((R + v)*d**2)

def mp_self_inductance_correction(mu, d, l, x):
    return mu*(mp_phi_selfL(x, d) + mp_phi_selfL(l, d) + 4*d) / (15*mp.pi)

def mp_self_inductance_outwire(mu, d, l, x):
    c1 = (x-d)*(mp_zeta(x-d,l-d) - mp_zeta(x-d,d))
    c2 = (l-d)*(mp_zeta(l-d,x-d) - mp_zeta(l-d,d))
    c3 = -d*(mp_zeta(d,l-d) + mp_zeta(d,x-d) - 2*mp_zeta(d,d))
    c4 = 2*((x-d)/mp_chi(x-d,l-d) + d/mp_chi(d,d) - (x-d)/mp_chi(x-d,d) - d/mp_chi(d,l-d))
    return mu*(c1+c2+c3+c4)/mp.pi

def mp_self_inductance_inwire(mu, d, l, x):
    D = mp_corner_flux(x, l, d)
    A1 = (mp_eta(x-d,d) - mp_eta(x-d,l-d) + mp_eta(x,l-d) - mp_eta(x,d)
          + 2*(mp_fnorm(x,l-d) + mp_fnorm(x-d,d) - mp_fnorm(x-d,l-d) - mp_fnorm(x,d)) + D/2)
    B1 = d*(mp_zeta(d,l-d) - mp_zeta(d,d)) + mp_fnorm(d,l-d) - mp_fnorm(d,d) + d - (l-d)
    C1 = (mp_fnorm(l-d,d)**3 - (l-d)**3)/(3*d**2) + (d - mp_fnorm(d,d)**3/d**2)/3
    A2 = (mp_eta(x-d,l) - mp_eta(x-d,l-d) + mp_eta(l-d,d) - mp_eta(l,d)
          + 2*(mp_fnorm(l-d,d) + mp_fnorm(x-d,l) - mp_fnorm(x-d,l-d) - mp_fnorm(l,d)) + D/2)
    B2 = d*(mp_zeta(d,x-d) - mp_zeta(d,d)) + mp_fnorm(x-d,d) - mp_fnorm(d,d) + d - (x-d)
    C2 = (mp_fnorm(x-d,d)**3 - (x-d)**3)/(3*d**2) + (d - mp_fnorm(d,d)**3/d**2)/3
    naive = mu*(A1+B1+C1+A2+B2+C2)/mp.pi
    return naive + mp_self_inductance_correction(mu, d, l, x)

def mp_self_inductance(mu, d, l, x):
    return mp_self_inductance_outwire(mu, d, l, x) + mp_self_inductance_inwire(mu, d, l, x)

max_reldiff = 0.0
for (xf, lf, df) in TEST_PARAMS:
    x, l, d = mp.mpf(xf), mp.mpf(lf), mp.mpf(df)
    eps = d * mp.mpf('1e-15')

    fd_D = (mp_corner_flux(x+eps, l, d) - mp_corner_flux(x-eps, l, d)) / (2*eps)
    an_D = mp.mpf(inductance.corner_flux(xf, lf, df))
    # dD/dx has no separate closed form in inductance.py; compare the
    # high-precision finite difference of corner_flux against itself at a
    # coarser step, as an internal convergence check (Richardson-style).
    eps2 = eps * 4
    fd_D2 = (mp_corner_flux(x+eps2, l, d) - mp_corner_flux(x-eps2, l, d)) / (2*eps2)
    reldiff_D = float(abs(fd_D - fd_D2) / abs(fd_D))

    fd_total = (mp_self_inductance(1, d, l, x+eps) - mp_self_inductance(1, d, l, x-eps)) / (2*eps)
    an_total = inductance.self_inductance_dx(1.0, df, lf, xf)
    reldiff = float(abs(mp.mpf(an_total) - fd_total) / abs(fd_total))
    max_reldiff = max(max_reldiff, reldiff, reldiff_D)
    print(f"  x={xf},l={lf},d={df}: dL/dx analytic={an_total:.12f}  mpmath-fd={float(fd_total):.12f}  reldiff={reldiff:.2e}"
          f"  (dD/dx step-convergence: {reldiff_D:.2e})")
report('A3 dL/dx vs. high-precision (mpmath) finite difference of L(x,l)', f'{max_reldiff:.2e}', '<1e-9', max_reldiff < 1e-9)


  x=63.0,l=37.0,d=1.0: dL/dx analytic=1.255363953165  mpmath-fd=1.255363953165  reldiff=8.88e-14  (dD/dx step-convergence: 9.85e-19)
  x=15.0,l=8.0,d=1.0: dL/dx analytic=0.763617831794  mpmath-fd=0.763617831794  reldiff=1.25e-14  (dD/dx step-convergence: 7.43e-23)
  x=500.0,l=90.0,d=2.0: dL/dx analytic=1.293844603243  mpmath-fd=1.293844603243  reldiff=1.37e-13  (dD/dx step-convergence: 2.52e-14)
[PASS] A3 dL/dx vs. high-precision (mpmath) finite difference of L(x,l): measured=1.37e-13  threshold=<1e-9  


### A4. Field integral along the bar (fully linked)

> "That identity needs one word of care. The bare field integral gives the
> gradient of the *fully linked* inductance $L_{fl}$" (body, Sec. 2.2);
> "the two corner integrals turn out to be equal to each other ...
> (verified numerically) ... this was checked against direct
> (high-precision) numerical integration of (\ref{bind2}) for several
> values of $x$, $l$ and $d$." (supplementary derivations, "Magnetic force
> on the free charges of the bar")

At $p=x$, the field is piecewise in $h$ over three strips. Check the two
corner-strip integrals are equal, and that their sum with the bulk strip
reproduces half the (unweighted, fully-linked) $dL/dx$. The linked
version, $\tfrac12\,dL/dx$ itself, is check A7.

In [5]:
max_reldiff = 0.0
for (x, l, d) in TEST_PARAMS:
    I_lort, _ = sciint.quad(lambda h: field.f_lo_rt(x, h, x, l, d), 0, d)
    I_uprt, _ = sciint.quad(lambda h: field.f_up_rt(x, h, x, l, d), l-d, l)
    reldiff_corners = abs(I_lort - I_uprt) / abs(I_lort)
    max_reldiff = max(max_reldiff, reldiff_corners)
    print(f"  x={x},l={l},d={d}: I_lo,rt={I_lort:.8f}  I_up,rt={I_uprt:.8f}  reldiff={reldiff_corners:.2e}")

print()
fully_linked_dLdx = lambda x, l, d: (
    inductance.self_inductance_dx(1.0, d, l, x) - inductance.self_inductance_correction_dx(1.0, d, x)
)
for (x, l, d) in TEST_PARAMS:
    I_lort, _ = sciint.quad(lambda h: field.f_lo_rt(x, h, x, l, d), 0, d)
    I_inrt, _ = sciint.quad(lambda h: field.f_in_rt(x, h, x, l, d), d, l-d)
    I_uprt, _ = sciint.quad(lambda h: field.f_up_rt(x, h, x, l, d), l-d, l)
    direct = I_lort + I_inrt + I_uprt
    closed = fully_linked_dLdx(x, l, d) / 2
    reldiff = abs(direct - closed) / abs(closed)
    max_reldiff = max(max_reldiff, reldiff)
    print(f"  x={x},l={l},d={d}: direct=int B_ind(x,h)dh={direct:.8f}  (1/2)(fully-linked dL/dx)={closed:.8f}  reldiff={reldiff:.2e}")
report('A4 bar-axis field integral (on-axis field vs. half of dL/dx)', f'{max_reldiff:.2e}', '~0 (quadrature precision)', max_reldiff < 1e-6)


  x=63.0,l=37.0,d=1.0: I_lo,rt=0.04231607  I_up,rt=0.04231607  reldiff=0.00e+00
  x=15.0,l=8.0,d=1.0: I_lo,rt=0.05179948  I_up,rt=0.05179948  reldiff=0.00e+00
  x=500.0,l=90.0,d=2.0: I_lo,rt=0.04160565  I_up,rt=0.04160565  reldiff=0.00e+00

  x=63.0,l=37.0,d=1.0: direct=int B_ind(x,h)dh=0.66746904  (1/2)(fully-linked dL/dx)=0.66746904  reldiff=8.23e-14
  x=15.0,l=8.0,d=1.0: direct=int B_ind(x,h)dh=0.42156823  (1/2)(fully-linked dL/dx)=0.42156823  reldiff=1.12e-14
  x=500.0,l=90.0,d=2.0: direct=int B_ind(x,h)dh=0.68671093  (1/2)(fully-linked dL/dx)=0.68671093  reldiff=1.19e-13
[PASS] A4 bar-axis field integral (on-axis field vs. half of dL/dx): measured=1.19e-13  threshold=~0 (quadrature precision)  


### A5. $\Delta L(x,l)$: weighted quadrature and the isolated-wire limit

> "it was verified independently by direct numerical quadrature of the
> weighted field integral over all four sides and four corners, matching
> to quadrature precision over a wide range of $x,l,d$, and by checking
> that an isolated straight wire's internal self-inductance...
> reproduces the classical energy-based result." (supplementary
> derivations, "Enclosed-current-fraction weighting")

In [6]:
max_reldiff = 0.0
for (x, l, d) in TEST_PARAMS:
    unweighted = total_over_8_regions(field.UNWEIGHTED, x, l, d)
    weighted = total_over_8_regions(field.WEIGHTED, x, l, d)
    direct_deltaL = weighted - unweighted
    closed = inductance.self_inductance_correction(1.0, d, l, x)
    reldiff = abs(closed - direct_deltaL) / abs(direct_deltaL)
    max_reldiff = max(max_reldiff, reldiff)
    print(f"  x={x},l={l},d={d}: closed(DeltaL)={closed:.8f}  direct(weighted-unweighted)={direct_deltaL:.8f}  reldiff={reldiff:.2e}")
report('A5 Delta L vs. weighted-quadrature difference', f'{max_reldiff:.2e}', '~0 (quadrature precision)', max_reldiff < 1e-4)

print()
print("Isolated straight wire: naive (fully-linked) flux vs. (r/d)^2-weighted vs. energy-based L/length")
mu, d = 1.0, 1.0
naive, _ = sciint.quad(lambda r: mu*r/(2*np.pi*d**2), 0, d)
weighted, _ = sciint.quad(lambda r: (r/d)**2 * mu*r/(2*np.pi*d**2), 0, d)
W, _ = sciint.quad(lambda r: ((mu*r/(2*np.pi*d**2))**2/(2*mu))*2*np.pi*r, 0, d)
L_energy = 2*W
print(f"  naive (fully-linked)   = {naive:.10f}   expect mu/(4 pi) = {mu/(4*np.pi):.10f}")
print(f"  (r/d)^2-weighted       = {weighted:.10f}   expect mu/(8 pi) = {mu/(8*np.pi):.10f}")
print(f"  energy-based           = {L_energy:.10f}   expect mu/(8 pi) = {mu/(8*np.pi):.10f}")
ok_isolated = (abs(weighted - mu/(8*np.pi)) < 1e-9 and abs(L_energy - mu/(8*np.pi)) < 1e-9
               and abs(naive - mu/(4*np.pi)) < 1e-9)
report('A5 isolated-wire limit: weighted/energy = mu/(8pi), naive = mu/(4pi)', 'match to 1e-9', 'match to 1e-9', ok_isolated)


  x=63.0,l=37.0,d=1.0: closed(DeltaL)=-7.87343346  direct(weighted-unweighted)=-7.87343346  reldiff=6.99e-15


  x=15.0,l=8.0,d=1.0: closed(DeltaL)=-1.74793755  direct(weighted-unweighted)=-1.74793755  reldiff=5.82e-14


  x=500.0,l=90.0,d=2.0: closed(DeltaL)=-46.78163847  direct(weighted-unweighted)=-46.78163847  reldiff=4.11e-13
[PASS] A5 Delta L vs. weighted-quadrature difference: measured=4.11e-13  threshold=~0 (quadrature precision)  

Isolated straight wire: naive (fully-linked) flux vs. (r/d)^2-weighted vs. energy-based L/length
  naive (fully-linked)   = 0.0795774715   expect mu/(4 pi) = 0.0795774715
  (r/d)^2-weighted       = 0.0397887358   expect mu/(8 pi) = 0.0397887358
  energy-based           = 0.0397887358   expect mu/(8 pi) = 0.0397887358
[PASS] A5 isolated-wire limit: weighted/energy = mu/(8pi), naive = mu/(4pi): measured=match to 1e-9  threshold=match to 1e-9  


### A6. $d\Delta L/dx$: the stable form vs. the naive (catastrophically-cancelling) form

> "checked against a finite-difference derivative of ($\Delta L$) to
> machine precision (using extended-precision arithmetic to rule out this
> same cancellation contaminating the finite-difference check itself)."
> (supplementary derivations, "Derivatives with respect to $x$")

In [7]:
def dDeltaL_dx_naive(mu, x, d=1.0):
    # algebraically identical to inductance.self_inductance_correction_dx,
    # but with the two near-cancelling O(x^3) terms left un-combined
    return mu*x*(3*d**2*x + 2*x**3 - 2*(d**2+x**2)**1.5) / (3*np.pi*d**4)

def mp_deltaL(x, l, d):
    def phi(v):
        R = mp.sqrt(d**2 + v**2)
        return (v*R*(v**2 - 2*d**2) - v**4 - 6*d**2*v**2 - 2*d**4) / ((R + v)*d**2)
    return (phi(x) + phi(l) + 4*d) / (15*mp.pi)

print("stable dDeltaL/dx  vs.  high-precision finite difference  vs.  naive float64 form:")
max_reldiff = 0.0
for xf in [63.0, 1e3, 1e4, 1e5, 1e6]:
    x, l, d = mp.mpf(xf), mp.mpf(100.0), mp.mpf(1.0)
    eps = x * mp.mpf('1e-10')
    fd = (mp_deltaL(x+eps, l, d) - mp_deltaL(x-eps, l, d)) / (2*eps)
    stable = inductance.self_inductance_correction_dx(1.0, 1.0, xf)
    naive = dDeltaL_dx_naive(1.0, xf)
    reldiff = float(abs(mp.mpf(stable) - fd) / abs(fd))
    max_reldiff = max(max_reldiff, reldiff)
    print(f"  x={xf:.0e}: high-prec-fd={float(fd):.12f}  stable={stable:.12f}  naive(float64)={naive:.12f}")
report('A6 stable dDeltaL/dx vs. high-precision finite difference', f'{max_reldiff:.2e}', '<1e-9', max_reldiff < 1e-9)
naive_fails = abs(dDeltaL_dx_naive(1.0, 1e5)) < 1e-8  # collapses to ~0 well before the true value
report('A6 naive float64 form demonstrably fails at large x (motivating the stable rewrite)',
       f'{dDeltaL_dx_naive(1.0, 1e5):.3e}', 'collapses to ~0 (bug demonstration, not a paper claim)', naive_fails)


stable dDeltaL/dx  vs.  high-precision finite difference  vs.  naive float64 form:
  x=6e+01: high-prec-fd=-0.079574130236  stable=-0.079574130236  naive(float64)=-0.079574130360
  x=1e+03: high-prec-fd=-0.079577458283  stable=-0.079577458283  naive(float64)=-0.079584352329
  x=1e+04: high-prec-fd=-0.079577471413  stable=-0.079577471413  naive(float64)=0.000000000000
  x=1e+05: high-prec-fd=-0.079577471545  stable=-0.079577471545  naive(float64)=0.000000000000
  x=1e+06: high-prec-fd=-0.079577471546  stable=-0.079577471546  naive(float64)=0.000000000000
[PASS] A6 stable dDeltaL/dx vs. high-precision finite difference: measured=2.53e-16  threshold=<1e-9  
[PASS] A6 naive float64 form demonstrably fails at large x (motivating the stable rewrite): measured=0.000e+00  threshold=collapses to ~0 (bug demonstration, not a paper claim)  


### A7. The half-boundary identity with the linked field

> "The striking feature is the middle term: the back-emf $L'vI$ is shared
> *precisely in half* between the two routes." (body, Sec. 3.1);
> "$\int_0^lf_B^{\rm link}(x,h;x)\,dh=\frac12\frac{dL}{dx}$, which holds
> *exactly*, for every $x$, $l$ and $d$ ... The identity was also checked
> by direct numerical quadrature of the left-hand side of
> (\ref{halfboundary}) against a high-precision numerical derivative of
> (\ref{Llink}), over a wide range of $x$, $l$ and $d$." (supplementary
> derivations, "The electromotive force: the flux rule, split in half")

On the bar's axis the linked field differs from the bare one only in the
two corner strips, where the near rail's own field carries the weight
$(r/d)^2$ (`field.f_lo_rt_w`, `field.f_up_rt_w`). The right-hand side is
`self_inductance_dx`, which A3 has already checked against an
extended-precision derivative of $L(x,l)$. This identity is what makes the
motional and the Faraday halves of $L'vI$ exactly equal at every bar
position, and the body's force (5) use $\frac12L'I$.

In [8]:
max_reldiff = 0.0
for (x, l, d) in TEST_PARAMS + [(500.0, 100.0, 1.0)]:
    I_lo, _ = sciint.quad(lambda h: field.f_lo_rt_w(x, h, x, l, d), 0, d)
    I_in, _ = sciint.quad(lambda h: field.f_in_rt(x, h, x, l, d), d, l-d)
    I_up, _ = sciint.quad(lambda h: field.f_up_rt_w(x, h, x, l, d), l-d, l)
    direct = I_lo + I_in + I_up
    half = inductance.self_inductance_dx(1.0, d, l, x) / 2
    reldiff = abs(direct - half) / abs(half)
    max_reldiff = max(max_reldiff, reldiff)
    print(f"  x={x},l={l},d={d}: int f_link(x,h;x) dh={direct:.12f}  "
          f"(1/2) dL/dx={half:.12f}  reldiff={reldiff:.2e}")
report('A7 linked field along the bar = (1/2) dL/dx (the half-split)',
       f'{max_reldiff:.2e}', '~0 (quadrature precision)', max_reldiff < 1e-9)

  x=63.0,l=37.0,d=1.0: int f_link(x,h;x) dh=0.627681976582  (1/2) dL/dx=0.627681976582  reldiff=8.74e-14
  x=15.0,l=8.0,d=1.0: int f_link(x,h;x) dh=0.381808915897  (1/2) dL/dx=0.381808915897  reldiff=1.26e-14
  x=500.0,l=90.0,d=2.0: int f_link(x,h;x) dh=0.646922301621  (1/2) dL/dx=0.646922301621  reldiff=1.27e-13
  x=500.0,l=100.0,d=1.0: int f_link(x,h;x) dh=0.774308136597  (1/2) dL/dx=0.774308136593  reldiff=4.84e-12
[PASS] A7 linked field along the bar = (1/2) dL/dx (the half-split): measured=4.84e-12  threshold=~0 (quadrature precision)  


### A8. The two-wire-line field along the bar

> "Equation (\ref{tworail}) is confirmed numerically against the full
> nine-region $f_B$: in the external span the pointwise relative error
> decreases like $1/x^2$ (about $10^{-3}$ at $x=10^3d$ and $10^{-5}$ at
> $x=10^4d$, for $l/d=100$)." (supplementary derivations, "Asymptotic
> route: the sliding bar as a two-wire line")

Compare the full field on the bar's axis, `field.f_B(x, h, x, l, d)`, with
the two semi-infinite rails of eq. (\ref{fbasym}), over the external span
$d<h<l-d$. In the two corner strips (\ref{fbasym}) keeps only the near
rail's internal field; the far rail adds $\mu_0/[4\pi(l-h)]$ there, which
changes the corner integrals by $O(d/l)$, the same order as the difference
between $\ln[(l-d)/d]$ and $\ln(l/d)$ neglected in (\ref{tworail-Lp}). The
second check confirms that,
with that term restored, the corners also converge as $1/x^2$.

In [9]:
l_a8, d_a8 = 100.0, 1.0
h_ext = np.linspace(1.5, 98.5, 60)
h_cor = np.linspace(0.05, 0.95, 7)
ext_err, cor_err = {}, {}
for x in [1e3, 1e4]:
    ext_err[x] = max(abs(field.f_B(x, h, x, l_a8, d_a8)
                         / ((1/h + 1/(l_a8-h)) / (4*np.pi)) - 1) for h in h_ext)
    cor_err[x] = max(abs(field.f_B(x, h, x, l_a8, d_a8)
                         / ((h/d_a8**2 + 1/(l_a8-h)) / (4*np.pi)) - 1) for h in h_cor)
    print(f"  x={x:.0e}: external span max rel. err = {ext_err[x]:.2e},  "
          f"corner strips (with the far rail) = {cor_err[x]:.2e}")
rate = ext_err[1e3] / ext_err[1e4]
report('A8 two-wire field, external span: ~1e-3 at x=1e3 d, ~1e-5 at x=1e4 d',
       f'{ext_err[1e3]:.1e}, {ext_err[1e4]:.1e} (ratio {rate:.0f})',
       'orders 1e-3 and 1e-5, ratio ~100 (1/x^2)',
       5e-4 < ext_err[1e3] < 5e-3 and 5e-6 < ext_err[1e4] < 5e-5 and 80 < rate < 120)
report('A8 corner strips converge as 1/x^2 once the far rail is kept',
       f'{cor_err[1e3]/cor_err[1e4]:.0f}', '~100', 80 < cor_err[1e3]/cor_err[1e4] < 120)

  x=1e+03: external span max rel. err = 1.25e-03,  corner strips (with the far rail) = 8.32e-04
  x=1e+04: external span max rel. err = 1.25e-05,  corner strips (with the far rail) = 8.34e-06
[PASS] A8 two-wire field, external span: ~1e-3 at x=1e3 d, ~1e-5 at x=1e4 d: measured=1.2e-03, 1.2e-05 (ratio 100)  threshold=orders 1e-3 and 1e-5, ratio ~100 (1/x^2)  
[PASS] A8 corner strips converge as 1/x^2 once the far rail is kept: measured=100  threshold=~100  


## Part B: body-level checks (main text)

The field-integral identities of the body's Secs. 2.2 and 3.1 (the fully
linked and the linked integral along the bar) are checks A4 and A7 above
and are not repeated here. B1-B10 follow the original numbering; B11-B16
check the remaining numbers of the body (Secs. 3.2-3.4 and 4.1) and the supplement's
statements about the runs behind them.

### B1. Asymptotic form of $L(x,l)$ vs. the exact $L(x,l)$

> "The full and asymptotic forms agree to better than $3\%$ already at
> $x=100d=l$." (body, Sec. 2.2); "At $x=100d$ the relative error of
> (\ref{Lasymp}) is less than $3\%$" (supplementary derivations,
> "Asymptotic behaviour of the self-inductance")

In [10]:
d = 1.0
x_test = np.array([50.0, 100.0, 200.0, 500.0, 2000.0])
exact = np.array([inductance.self_inductance(1.0, d, 100.0, xx) for xx in x_test])
asym = np.array([inductance.self_inductance_asymp(1.0, d, 100.0, xx) for xx in x_test])
relerr = np.abs(exact - asym) / exact
for xx, e, a, r in zip(x_test, exact, asym, relerr):
    print(f"  x={xx:.0f} (x/d={xx/d:.0f}): exact={e:.4f}  Lasymp={a:.4f}  relerr={r:.3%}")
err_at_100d = relerr[x_test == 100.0][0]
report('B1 asymptotic form error at x=100d', f'{err_at_100d:.2%}', '<3%', err_at_100d < 0.03)


  x=50 (x/d=50): exact=176.4608  Lasymp=190.7276  relerr=8.085%
  x=100 (x/d=100): exact=260.3228  Lasymp=268.0001  relerr=2.949%
  x=200 (x/d=200): exact=418.6053  Lasymp=422.5449  relerr=0.941%
  x=500 (x/d=500): exact=884.5905  Lasymp=886.1795  relerr=0.180%
  x=2000 (x/d=2000): exact=3203.9547  Lasymp=3204.3526  relerr=0.012%
[PASS] B1 asymptotic form error at x=100d: measured=2.95%  threshold=<3%  


### B2. Exact asymptotic slope match with Rosa/Grover

> "As $x\to\infty$, (\ref{Lrosa}) has the asymptotic slope
> $\frac{\mu_0}{\pi}(\ln|l/d|+\frac14)$, agreeing exactly, for any $l/d$,
> with (\ref{dLasymp}) -- a nontrivial cross-check of the
> enclosed-current-fraction term $\Delta L$: without it, (\ref{Lxl})'s
> asymptotic slope would be ... larger than Rosa and Grover's by exactly
> $\mu_0/(4\pi)$" (supplementary derivations, "Asymptotic behaviour of the
> self-inductance", "Independent cross-check")

In [11]:
for l, d in [(100.0, 1.0), (50.0, 1.0), (200.0, 2.0)]:
    lasymp_slope = inductance.self_inductance_dx_asymp(1.0, d, l)
    rosa_grover_slope = (1.0/np.pi)*(np.log(np.abs(l/d)) + 0.25)  # slope of the Rosa/Grover asymptotic form
    diff = abs(lasymp_slope - rosa_grover_slope)
    print(f"  l={l},d={d}: L_asymp slope={lasymp_slope:.10f}  Rosa/Grover slope={rosa_grover_slope:.10f}  diff={diff:.2e}")

report('B2 asymptotic slope match (this work vs. Rosa/Grover)', f'{diff:.2e}', '~0 (exact, analytic)', diff < 1e-12)

# the "without Delta L, slope exceeds by mu0/(4pi)" half of the claim
l, d = 100.0, 1.0
naive_slope = (1.0/np.pi)*(np.log(np.abs(l/d)) + 0.5)  # naive (unweighted) slope, no Delta L
excess = naive_slope - lasymp_slope
report('B2 naive (no Delta L) slope exceeds by mu0/(4pi)', f'{excess:.10f}', f'{1/(4*np.pi):.10f}', abs(excess - 1/(4*np.pi)) < 1e-9)


  l=100.0,d=1.0: L_asymp slope=1.5454486693  Rosa/Grover slope=1.5454486693  diff=0.00e+00
  l=50.0,d=1.0: L_asymp slope=1.3248130692  Rosa/Grover slope=1.3248130692  diff=0.00e+00
  l=200.0,d=2.0: L_asymp slope=1.5454486693  Rosa/Grover slope=1.5454486693  diff=0.00e+00
[PASS] B2 asymptotic slope match (this work vs. Rosa/Grover): measured=0.00e+00  threshold=~0 (exact, analytic)  
[PASS] B2 naive (no Delta L) slope exceeds by mu0/(4pi): measured=0.0795774715  threshold=0.0795774715  


### B3. Comparison with Rosa/Grover at finite $x$

> "the two are visually indistinguishable over the range plotted, and agree
> to within $0.5\%$ already at $x=100d$ (with $l/d=100$)." (supplementary
> derivations, "Independent cross-check")

The agreement at $x=2d$ is not quoted in the text; it is reported here as a
*supporting* check of how far the agreement extends.

In [12]:
def rosa_inductance(mu, l, x, d):
    r = np.sqrt(x**2 + l**2)
    return (mu/np.pi)*(x*np.log(np.abs(2*x/d)) + l*np.log(np.abs(2*l/d))
              - 0.75*(x+l) - x*np.log(np.abs((r+x)/l)) + r - l
              - l*np.log(np.abs((r+l)/x)) + r - x)

def dL_rosa(mu, l, x, d):
    r = np.sqrt(x**2 + l**2)
    return (mu/np.pi)*(np.log(np.abs(2*x/d)) - np.log(np.abs((r+x)/l)) - 0.75
                        + x/r - l*x/(r*(r+l)) + l/x)

mu, l, d = 1.0, 100.0, 1.0
x_arr = np.arange(2, 501)
L_eq = np.array([inductance.self_inductance(mu, d, l, xx) for xx in x_arr])
L_rosa_arr = rosa_inductance(mu, l, x_arr, d)
dL_eq = np.array([inductance.self_inductance_dx(mu, d, l, xx) for xx in x_arr])
dL_rosa_arr = dL_rosa(mu, l, x_arr, d)

i100, i2 = np.argmin(np.abs(x_arr-100)), np.argmin(np.abs(x_arr-2))
pct_100d = abs(1 - L_rosa_arr[i100]/L_eq[i100])
pct_2d_L = abs(1 - L_rosa_arr[i2]/L_eq[i2])
pct_2d_dL = abs(1 - dL_rosa_arr[i2]/dL_eq[i2])
print(f"  |1 - L_rosa/L| at x=100d, l/d=100:  {pct_100d:.4%}")
print(f"  |1 - L_rosa/L| at x=2d,   l/d=100:  {pct_2d_L:.4%}")
print(f"  |1 - dLrosa/dL| at x=2d, l/d=100:   {pct_2d_dL:.4%}")

report('B3 L agreement at x>=100d, l/d=100', f'{pct_100d:.4%}', '<0.5%', pct_100d < 0.005)
report('B3 (supporting) L agreement at x=2d, l/d=100', f'{pct_2d_L:.4%}', '<2%', pct_2d_L < 0.02)
report('B3 (supporting) dL/dx agreement at x=2d, l/d=100', f'{pct_2d_dL:.4%}', '<0.1%', pct_2d_dL < 0.001)


  |1 - L_rosa/L| at x=100d, l/d=100:  0.1952%
  |1 - L_rosa/L| at x=2d,   l/d=100:  1.5951%
  |1 - dLrosa/dL| at x=2d, l/d=100:   0.0770%
[PASS] B3 L agreement at x>=100d, l/d=100: measured=0.1952%  threshold=<0.5%  
[PASS] B3 (supporting) L agreement at x=2d, l/d=100: measured=1.5951%  threshold=<2%  
[PASS] B3 (supporting) dL/dx agreement at x=2d, l/d=100: measured=0.0770%  threshold=<0.1%  


### B4. Finite-radius strip-force offset

> "the strip average of the other three segments' field exceeds the on-axis
> evaluation (\ref{bind3}) by a constant offset $\kappa\mu_0$ with
> $\kappa\approx0.0438$, found numerically to be universal in the thin-wire
> regime -- independent of $x$, $l$ and $d$ for $d\ll x,l$. Relative to the
> on-axis integral it corrects, $\tfrac12dL_{fl}/dx$, this is a $\sim5\%$
> correction for the numerical examples ($l/d=100$, $x_0=500$)"
> (supplementary derivations, "Magnetic force on the free charges of the bar")

Integrate the Lorentz force over the bar's actual (finite) cross
section, rather than evaluating the field only on its axis, and compare
the resulting strip average to the on-axis value the paper's force
expressions use, at the numerical example and at several other
$(x,l,d)$ to test the universality of $\kappa$.

In [13]:
def _g(v, l, d):
    return inductance.zeta(v, l-d) - inductance.zeta(v, d) + 1/inductance.chi(v, l-d) - 1/inductance.chi(v, d)

def J_bulk_cross(p, x, l, d):
    a = x - p
    return (1/(2*np.pi)) * (_g(p, l, d) + inductance.zeta(a, l-d) - inductance.zeta(a, d))

def J_corner_cross(p, x, l, d):
    a = x - p
    termA = (1/(4*np.pi*d**2)) * (a**2/inductance.chi(a, d) - a*abs(a))
    termB = (p/(4*np.pi*d**2)) * (d/inductance.chi(d, p) - p)
    termC = (1/(4*np.pi)) * (inductance.zeta(p, l) - inductance.zeta(p, l-d) + inductance.zeta(a, l) - inductance.zeta(a, l-d))
    termD = (1/(4*np.pi)) * (1/inductance.chi(p, d) - 1 + 1/inductance.chi(p, l) - 1/inductance.chi(p, l-d))
    return termA + termB + termC + termD

def strip_avg_cross(x, l, d):
    left, _ = sciint.quad(lambda p: J_bulk_cross(p, x, l, d) + 2*J_corner_cross(p, x, l, d),
                           x-d+1e-13, x-1e-13, epsabs=1e-13, epsrel=1e-13, limit=200)
    def J_gt(p, x, l, d):
        a = x - p
        return (1/(2*np.pi)) * (1/inductance.chi(p, l) + inductance.zeta(p, l) + inductance.zeta(a, l) - 1
                                 - np.log((p-x)/p))
    right, _ = sciint.quad(J_gt, x+1e-13, x+d-1e-13, args=(x, l, d), epsabs=1e-13, epsrel=1e-13, limit=200)
    return (left + right) / (2*d)

def onaxis_half(x, l, d):
    return (inductance.self_inductance_dx(1.0, d, l, x)
            - inductance.self_inductance_correction_dx(1.0, d, x)) / 2

x, l, d = 500.0, 100.0, 1.0
kappas = []
for (xx, ll, dd) in [(500.0, 100.0, 1.0), (50.0, 100.0, 1.0), (500.0, 50.0, 1.0),
                     (500.0, 200.0, 1.0), (500.0, 100.0, 0.5), (2000.0, 400.0, 1.0)]:
    kappas.append(strip_avg_cross(xx, ll, dd) - onaxis_half(xx, ll, dd))
    print(f"  x={xx:6.0f}, l={ll:5.0f}, d={dd}: kappa = {kappas[-1]:.6f}")
oh = onaxis_half(x, l, d)
frac = kappas[0] / oh
print(f"  at x0=500, l/d=100: (1/2) dL_fl/dx [on-axis] = {oh:.6f}, kappa/that = {frac:.2%}")
report('B4 kappa = 0.0438, universal in x, l, d', f'{min(kappas):.5f} to {max(kappas):.5f}',
       '0.0438, spread < 0.1%', round(kappas[0], 4) == 0.0438 and max(kappas) / min(kappas) - 1 < 1e-3)
report('B4 offset ~5% of (1/2) dL_fl/dx at l/d=100, x0=500', f'{frac:.2%}', '5% (1 sig. fig.)',
       round(100 * frac) == 5)

# self-force cancellation (bar exerts no net force on itself, by symmetry)
def B_bar(p, h, x, l, d):
    pm = x - p
    inside = (0.0 <= h <= l) and (abs(pm) < d)
    if inside:
        return (1/(4*np.pi*d**2)) * pm * (inductance.chi(l-h, pm) + inductance.chi(h, pm))
    return (1/(4*np.pi*pm)) * (inductance.chi(l-h, pm) + inductance.chi(h, pm))

max_asym = 0.0
for delta in np.linspace(1e-4, d-1e-4, 8):
    for h in np.linspace(1e-3, l-1e-3, 8):
        max_asym = max(max_asym, abs(B_bar(x+delta, h, x, l, d) + B_bar(x-delta, h, x, l, d)))
report('B4 bar self-force cancellation (odd in p about the bar axis)', f'{max_asym:.2e}', '~0 (exact, by symmetry)', max_asym < 1e-9)


  x=   500, l=  100, d=1.0: kappa = 0.043780
  x=    50, l=  100, d=1.0: kappa = 0.043808
  x=   500, l=   50, d=1.0: kappa = 0.043780
  x=   500, l=  200, d=1.0: kappa = 0.043780
  x=   500, l=  100, d=0.5: kappa = 0.043780
  x=  2000, l=  400, d=1.0: kappa = 0.043780
  at x0=500, l/d=100: (1/2) dL_fl/dx [on-axis] = 0.814097, kappa/that = 5.38%
[PASS] B4 kappa = 0.0438, universal in x, l, d: measured=0.04378 to 0.04381  threshold=0.0438, spread < 0.1%  
[PASS] B4 offset ~5% of (1/2) dL_fl/dx at l/d=100, x0=500: measured=5.38%  threshold=5% (1 sig. fig.)  
[PASS] B4 bar self-force cancellation (odd in p about the bar axis): measured=0.00e+00  threshold=~0 (exact, by symmetry)  


### B5. Second-order convergence and energy conservation (supporting)

> "integrated with a second-order velocity-Verlet scheme at a timestep
> $\Delta t=2\times10^{-4}$" (body, Sec. 3.3); "both choices being what keep
> the whole update second order" and the energy "is exactly conserved by
> (\ref{eq:vel-eom})--(\ref{eq:curr-eom}) ... giving a convenient,
> quadrature-free diagnostic for the accuracy of a numerical integrator."
> (supplementary derivations, "Velocity-Verlet integration scheme")

Runs the integrator at several step sizes and fits the observed
convergence order of the relative energy error at $\rho=0$ (where the
continuum equations conserve $\frac12Mv^2+\frac12LI^2$ exactly, isolating
the integrator's own error), and reports the error at the production step
$\Delta t=2\times10^{-4}$ of the body's Fig. 2.

In [14]:
l, x0, B0 = 100.0, 500.0, 0.3
p0 = -2.e-2
T_total = 20.0
dt_list = [2e-3, 1e-3, 5e-4, 2.5e-4, 1.25e-4]
E0 = p0**2/2

max_errs = []
for dt in dt_list:
    n_steps = int(T_total/dt)
    n_print = max(1, n_steps//2000)
    x_, p_, I_, F_, K_, Eb_, Q_ = dynamics.verlet_integrate(x0, p0, l, B0, 0.0, dt, n_steps, n_print)
    Etot = np.array(K_) + np.array(Eb_)
    rel_err = np.abs((Etot - E0)/E0)
    max_errs.append(np.max(rel_err))
    print(f"  dt={dt:.2e}: max|rel. energy err|={max_errs[-1]:.3e}")

order = np.polyfit(np.log(dt_list), np.log(max_errs), 1)[0]
print(f"  fitted convergence order: {order:.3f}")
report('B5 convergence order', f'{order:.3f}', '~2.0', 1.8 < order < 2.2)

# production step size
dt_prod = 2.e-4
n_steps = int(T_total/dt_prod)
n_print = max(1, n_steps//2000)
x_, p_, I_, F_, K_, Eb_, Q_ = dynamics.verlet_integrate(x0, p0, l, B0, 0.0, dt_prod, n_steps, n_print)
Etot = np.array(K_) + np.array(Eb_)
rel_err_prod = np.max(np.abs((Etot - E0)/E0))
print(f"  at production dt=2e-4: max|rel. energy err|={rel_err_prod:.3e}")
report('B5 energy error at production dt=2e-4', f'{rel_err_prod:.2e}', '<1e-6', rel_err_prod < 1e-6)


  dt=2.00e-03: max|rel. energy err|=1.017e-06


  dt=1.00e-03: max|rel. energy err|=2.544e-07
  dt=5.00e-04: max|rel. energy err|=6.359e-08


  dt=2.50e-04: max|rel. energy err|=1.590e-08


  dt=1.25e-04: max|rel. energy err|=3.976e-09
  fitted convergence order: 2.000
[PASS] B5 convergence order: measured=2.000  threshold=~2.0  


  at production dt=2e-4: max|rel. energy err|=1.018e-08
[PASS] B5 energy error at production dt=2e-4: measured=1.02e-08  threshold=<1e-6  


### B6. Over-/under-damped crossover field $B_{0,c}(\rho)$

> "The over- and under-damped regimes meet at the critical field
> $B_{0,c}(\rho)=\frac{\sqrt{ML(x_0)}}{l}\gamma(\rho)$, where
> $\omega_0=\gamma$, with $L$ evaluated at $x_0$ as just discussed."
> (body, Sec. 3.2.1)

Checks the defining property directly: at $B_0=B_{0,c}(\rho)$, the
damped-oscillator model's predicted frequency $\omega_d=\sqrt{\omega_0^2-
\gamma^2}$ should vanish (the critically-damped point), and should be
real and positive just above $B_{0,c}$. Both $\omega_0$ and
$\gamma=R/(2L)$ are evaluated at $x_0$ (`models.gamma_exact`), as the
body does; for $\rho=1$ this gives $B_{0,c}=0.0642$, so that
$B_0=2\times10^{-2}$ is $Q=0.16$, as in Fig. 2.

In [15]:
l, x0 = 100.0, 500.0
for rho in [0.3, 0.6, 1.0]:
    B0c = models.B0_critical(l, x0, rho)
    omega_d_at = models.omega_predicted(l, x0, B0c, rho)
    omega_d_above = models.omega_predicted(l, x0, B0c*1.2, rho)
    omega_d_below = models.omega_predicted(l, x0, B0c*0.8, rho)
    print(f"  rho={rho}: B0c={B0c:.5f}  omega_d(B0c)={omega_d_at:.2e}  "
          f"omega_d(1.2*B0c)={omega_d_above:.5f}  omega_d(0.8*B0c)={omega_d_below:.5f}")

ok = (omega_d_at < 1e-8) and (omega_d_above > 0) and (omega_d_below == 0.0)
report('B6 B0c is exactly the omega_d=0 crossover', f'omega_d(B0c)={omega_d_at:.2e}', '~0', ok)


  rho=0.3: B0c=0.01926  omega_d(B0c)=0.00e+00  omega_d(1.2*B0c)=0.04296  omega_d(0.8*B0c)=0.00000
  rho=0.6: B0c=0.03853  omega_d(B0c)=0.00e+00  omega_d(1.2*B0c)=0.08593  omega_d(0.8*B0c)=0.00000
  rho=1.0: B0c=0.06421  omega_d(B0c)=3.73e-09  omega_d(1.2*B0c)=0.14321  omega_d(0.8*B0c)=0.00000
[PASS] B6 B0c is exactly the omega_d=0 crossover: measured=omega_d(B0c)=3.73e-09  threshold=~0  


### B7. The rescaled expansion in $\varepsilon$

> "Measuring displacement, current and time on the scales of the motion ...
> In these variables the exact system (\ref{eom}) becomes (term by term in
> the supplement) ... Setting
> $\varepsilon=m=0$ leaves $\chi'''+\chi''/Q+\chi'=0$: the damped oscillator
> of Sec. 3.2 in rescaled time, with no free amplitude --- the dashed curves
> of Fig. 3, and the solution $(\chi_0,j_0)$ about which one expands."
> (body, Sec. 3.4)

Three things are asserted there, and each is checked below.

1. **The $\varepsilon\to0$ limit is the damped oscillator.** Integrating
   eqs. (`chieq`)-(`rescaled`) at $\varepsilon=0$ must reproduce the closed
   form `models.x_damped_oscillator_norm`, which was obtained independently
   by integrating $v(t)$ analytically.
2. **Eqs. (`chieq`)-(`rescaled`) are exact through $O(\varepsilon)$.** They
   linearise $L(x)$ and $R(x)$ in the excursion and keep nothing beyond, so
   their departure from the full simulation must fall as $\varepsilon^2$ --
   a fitted log-log slope of $2$, not $1$ (which would mean the
   $O(\varepsilon)$ terms were wrong) and not $3$ (which would mean they are
   accidentally better than claimed).
3. **The first correction is $O(\varepsilon)$.** The departure of the
   full simulation from the $\varepsilon$-independent $\chi_0$ must fall
   as $\varepsilon^1$, which is what makes $\chi_1$ the first correction
   and justifies calling the two signatures first order in it.

A note on timing: `verlet_integrate` records the state *after* each step,
so sample $i$ sits at $t=\Delta t\,(n_{print}i+1)$, one step later than
the `n_print*dt*arange` used elsewhere in these notebooks. The offset is
irrelevant to every quantity the paper reports (the drift is an
asymptote; the ring-down frequency comes from differences of crossing
times, which a constant shift cancels), but here it would put a floor of
$\omega_0\Delta t\approx10^{-3}$ under check 2 and hide the
$\varepsilon^2$ scaling, so it is corrected.

In [16]:
from scipy.integrate import solve_ivp

l, x0, B0, rho, dt_b7 = 100.0, 500.0, 0.03, 0.02, 1.0e-2
mu_b7 = d_b7 = 1.0
L_x0 = inductance.self_inductance(mu_b7, d_b7, l, x0)
Lp_x0 = inductance.self_inductance_dx(mu_b7, d_b7, l, x0)
w0_b7 = models.omega0(l, x0, B0)
T0_b7 = 2 * np.pi / w0_b7
Q_b7 = models.quality_factor(l, x0, B0, rho)
# eps/m = L'(x0+l)/L(x0) is fixed by the geometry alone, so m follows from eps
eps_over_m = Lp_x0 * (l + x0) / L_x0
print(f"  eps/m = L'(x0+l)/L(x0) = {eps_over_m:.4f},  Q = {Q_b7:.3f}")


def chi_rescaled(eps, s):
    """Eqs. (`chieq`)-(`rescaled`), integrated to high tolerance.
    chi'' = -j + (eps/2) j^2
    (1 + eps chi) j' = chi'(1 - eps j) - (1 + m chi) j / Q
    Launched towards the resistive end, so chi(0)=0, chi'(0)=-1, j(0)=0."""
    m = eps / eps_over_m

    def rhs(_, y):
        chi, chip, j = y
        return [chip,
                -j + 0.5 * eps * j**2,
                (chip * (1 - eps * j) - (1 + m * chi) * j / Q_b7) / (1 + eps * chi)]

    sol = solve_ivp(rhs, [0.0, s[-1]], [0.0, -1.0, 0.0],
                    dense_output=True, rtol=1e-11, atol=1e-13)
    return sol.sol(s)[0]


# --- 1. the eps -> 0 limit is the damped oscillator ---
s_lim = np.linspace(0.0, 6 * 2 * np.pi, 3000)
dev0 = np.max(np.abs(chi_rescaled(0.0, s_lim)
                     - models.x_damped_oscillator_norm(s_lim / w0_b7, l, x0, B0, rho)))
print(f"  eps=0 limit of eqs. (`chieq`)-(`rescaled`) vs closed form: max|dchi| = {dev0:.2e}")
report('B7 eqs. (`chieq`)-(`rescaled`) at eps=0 are the damped oscillator',
       f'{dev0:.1e}', '<1e-8', dev0 < 1e-8)

# --- 2 and 3. scaling of the two departures with eps ---
eps_b7 = np.array([0.05, 0.10, 0.20, 0.40])
n_b7 = int(6 * T0_b7 / dt_b7)
dev_resc, dev_0 = [], []
for eps in eps_b7:
    xb, *_ = dynamics.verlet_integrate(
        x0, -models.p0_from_epsilon(eps, l, x0, B0), l, B0, rho, dt_b7, n_b7, 10)
    t_b7 = dt_b7 * (10 * np.arange(len(xb)) + 1)      # see the timing note above
    chi_sim = (np.array(xb) - x0) / models.free_amplitude(eps, l, x0)
    dev_resc.append(np.max(np.abs(chi_sim - chi_rescaled(eps, t_b7 * w0_b7))))
    dev_0.append(np.max(np.abs(chi_sim - models.x_damped_oscillator_norm(
        t_b7, l, x0, B0, rho))))
dev_resc, dev_0 = np.array(dev_resc), np.array(dev_0)

slope_resc = np.polyfit(np.log(eps_b7), np.log(dev_resc), 1)[0]
slope_0 = np.polyfit(np.log(eps_b7), np.log(dev_0), 1)[0]
print(f"  {'eps':>6} {'|chi_sim - chi_resc|':>21} {'|chi_sim - chi_0|':>19}")
for e, a, b in zip(eps_b7, dev_resc, dev_0):
    print(f"  {e:6.2f} {a:21.3e} {b:19.4f}")
print(f"  fitted log-log slopes: rescaled-system residual {slope_resc:.2f}, "
      f"departure from chi_0 {slope_0:.2f}")

report('B7 eqs. (`chieq`)-(`rescaled`) are exact through O(eps) (residual ~ eps^2)',
       f'slope={slope_resc:.2f}', '2.0 +/- 0.2', 1.8 < slope_resc < 2.2)
report('B7 first correction to chi_0 is O(eps)',
       f'slope={slope_0:.2f}', '1.0 +/- 0.2', 0.8 < slope_0 < 1.2)


  eps/m = L'(x0+l)/L(x0) = 1.0504,  Q = 11.680
  eps=0 limit of eqs. (`chieq`)-(`rescaled`) vs closed form: max|dchi| = 1.07e-11
[PASS] B7 eqs. (`chieq`)-(`rescaled`) at eps=0 are the damped oscillator: measured=1.1e-11  threshold=<1e-8  


     eps  |chi_sim - chi_resc|   |chi_sim - chi_0|
    0.05             2.105e-05              0.0953
    0.10             8.358e-05              0.1880
    0.20             3.356e-04              0.3646
    0.40             1.323e-03              0.6803
  fitted log-log slopes: rescaled-system residual 1.99, departure from chi_0 0.95
[PASS] B7 eqs. (`chieq`)-(`rescaled`) are exact through O(eps) (residual ~ eps^2): measured=slope=1.99  threshold=2.0 +/- 0.2  
[PASS] B7 first correction to chi_0 is O(eps): measured=slope=0.95  threshold=1.0 +/- 0.2  


### B8. The dissipationless invariants and the turning points

> "Without dissipation, $Q^{-1}=0$, energy conservation reads
> $\chi'^2+(1+\varepsilon\chi)j^2=1$ ... $(1+\varepsilon\chi)j=\chi$,
> $\chi_\pm=\frac{\varepsilon\pm\sqrt{\varepsilon^2+4}}{2}$ ... The motion
> stays bounded for any $\varepsilon$, but its centre is displaced by
> $\varepsilon/2$" (body, Sec. 3.4); "at $\chi_-$ the current overshoots $I_m$
> by the factor $(1+\varepsilon\chi_-)^{-1/2}$" (supplementary derivations,
> "Lossless invariants")

At $\rho=0$ the rescaled system has two invariants: the energy,
$\chi'^2+(1+\varepsilon\chi)j^2=1$, and the stronger flux invariant
$(1+\varepsilon\chi)j=\chi$, which together give the orbit
$\chi'^2+\chi^2/(1+\varepsilon\chi)=1$ and hence the turning points
$\chi_\pm=(\varepsilon\pm\sqrt{\varepsilon^2+4})/2$ and the peak current
$|j|_{max}=(1+\varepsilon\chi_-)^{-1/2}$. All four predictions are read
off a dissipationless run of the *full* equations of motion below, so
they also test the linearisation of $L(x)$ that eq. (`LRexp`) makes: the
residual mismatch, a few parts in $10^4$ at $\varepsilon=0.4$, is the
curvature of the exact $L(x)$ over the swing, not an error in the
invariants.

In [17]:
eps_b8 = np.array([0.02, 0.10, 0.20, 0.30, 0.40, 0.50])
dt_b8, n_b8 = 1.0e-2, int(6 * T0_b7 / 1.0e-2)
worst_turn, worst_centre, worst_j = 0.0, 0.0, 0.0
print(f"  {'eps':>6} {'chi_-':>9} {'pred':>9} {'chi_+':>9} {'pred':>9} "
      f"{'centre':>8} {'eps/2':>7} {'|j|max':>8} {'pred':>8}")
for eps in eps_b8:
    Xm = models.free_amplitude(eps, l, x0)
    Im = eps * l * B0 / Lp_x0                      # = |p0|/sqrt(L(x0)), eq. (thresh)
    xb, pb, Ib, *_ = dynamics.verlet_integrate(
        x0, -models.p0_from_epsilon(eps, l, x0, B0), l, B0, 0.0, dt_b8, n_b8, 1)
    chi = (np.array(xb) - x0) / Xm
    j = np.array(Ib) / Im
    cm, cp = chi.min(), chi.max()
    pm, pp = (eps - np.sqrt(eps**2 + 4)) / 2, (eps + np.sqrt(eps**2 + 4)) / 2
    jmax_pred = 1 / np.sqrt(1 + eps * pm)
    worst_turn = max(worst_turn, abs(cm - pm), abs(cp - pp))
    worst_centre = max(worst_centre, abs(0.5 * (cm + cp) - eps / 2))
    worst_j = max(worst_j, abs(np.abs(j).max() - jmax_pred))
    print(f"  {eps:6.2f} {cm:9.4f} {pm:9.4f} {cp:9.4f} {pp:9.4f} "
          f"{0.5*(cm+cp):8.4f} {eps/2:7.4f} {np.abs(j).max():8.4f} {jmax_pred:8.4f}")

report('B8 turning points chi_+- of eq. (`fluxinv`)', f'{worst_turn:.1e}',
       '<1e-3 (exact-L curvature)', worst_turn < 1e-3)
report('B8 centre of the dissipationless orbit displaced by eps/2',
       f'{worst_centre:.1e}', '<1e-3', worst_centre < 1e-3)
report('B8 peak current (1+eps chi_-)^{-1/2}', f'{worst_j:.1e}', '<1e-3',
       worst_j < 1e-3)


     eps     chi_-      pred     chi_+      pred   centre   eps/2   |j|max     pred
    0.02   -0.9900   -0.9900    1.0100    1.0100   0.0100  0.0100   1.0101   1.0100
    0.10   -0.9512   -0.9512    1.0512    1.0512   0.0500  0.0500   1.0513   1.0512
    0.20   -0.9049   -0.9050    1.1049    1.1050   0.1000  0.1000   1.1050   1.1050
    0.30   -0.8611   -0.8612    1.1611    1.1612   0.1500  0.1500   1.1613   1.1612


    0.40   -0.8196   -0.8198    1.2196    1.2198   0.2000  0.2000   1.2201   1.2198


    0.50   -0.7805   -0.7808    1.2805    1.2808   0.2500  0.2500   1.2813   1.2808
[PASS] B8 turning points chi_+- of eq. (`fluxinv`): measured=3.1e-04  threshold=<1e-3 (exact-L curvature)  
[PASS] B8 centre of the dissipationless orbit displaced by eps/2: measured=2.0e-05  threshold=<1e-3  
[PASS] B8 peak current (1+eps chi_-)^{-1/2}: measured=5.1e-04  threshold=<1e-3  


### B9. The over-damped resting position

> "Below the critical point the balance can be integrated exactly ...
> $\chi_\infty=-(1-e^{-m/Q})/m$ ... Numerically (\ref{eq:rest-od}) tracks
> the rescaled system to better than $1.3\%$ over
> $0.01\le\varepsilon\le0.12$ at $Q=0.156$." (supplementary derivations,
> "Check: the over-damped limit in closed form"); "at $Q=0.16$ $c=20.4$
> ... both values are the slopes of the simulated $\chi_\infty(\varepsilon)$
> as $\varepsilon\to0$" (body, Sec. 3.4)

Below the critical point the bar does not ring down but glides to a stop,
and the shift of its resting position is mostly the fall of $R$ as it
advances. Eq. (`eq:rest-od`) is the closed form for that, obtained in the
quasi-static ($L\dot I\to0$) limit; its expansion gives only the $Q^{-2}$
part of $c$, $1/[2(\varepsilon/m)Q^2]=19.6$. The linear law (`restlaw`),
$c=1/4+(m/2\varepsilon)(1+Q^{-2})$, exact to first order at every $Q$,
adds the thrust's $1/4$ and the inductive $m/2\varepsilon$ to give
$20.35$.

The second check compares that with the slope of the simulated
$\chi_\infty(\varepsilon)$ *as $\varepsilon\to0$*. At finite $\varepsilon$
the slope is visibly lower ($\simeq19.93$ at $\varepsilon=0.01$), because
the higher powers of $m/Q$ that eq. (`eq:rest-od`) resums already bend the
curve, so the slope is measured at $\varepsilon=10^{-3}$ and
$2\times10^{-3}$ and Richardson-extrapolated to $\varepsilon=0$. For
contrast the cell also prints the two-limit value $1/4+1/[2(\varepsilon/m)Q^2]
=19.88$, which drops the $m/2\varepsilon$ term and is off by $2\%$ ($Q^2$)
at this $Q$. Every run is integrated for $20/\alpha$, by which the bar has
stopped to $\sim10^{-9}X_m$; at $10/\alpha$ the residual creep would still
bias the slope at $\varepsilon=10^{-3}$ by about $1\%$.

The third, a *supporting* check, shows that what the closed form leaves
out (the thrust and the inductive terms) is proportional to
$\varepsilon Q$, with coefficient $0.72$, over a factor of seven in $Q$ and
four in $\varepsilon$.


In [18]:
rho_b9, eps_b9 = 1.5, np.array([0.01, 0.02, 0.04, 0.08, 0.12])
eps_over_m_b9 = Lp_x0 * (l + x0) / L_x0        # geometry alone, eq. (LRexp)

def overdamped_scan(rho, eps_list=eps_b9):
    """Resting position chi_inf from the full equations, against eq.
    (`eq:rest-od`), for a glide launched towards the resistive end."""
    Q = models.quality_factor(l, x0, B0, rho)
    alpha = models.omega0(l, x0, B0) * Q        # textbook rate, = w0 Q
    out = []
    for eps in eps_list:
        m = eps / eps_over_m_b9
        xb, *_ = dynamics.verlet_integrate(
            x0, -models.p0_from_epsilon(eps, l, x0, B0), l, B0, rho,
            1.0e-2, int(20 / alpha / 1.0e-2), 10)
        chi = (xb[-1] - x0) / models.free_amplitude(eps, l, x0)
        out.append((eps, m, chi, -(1 - np.exp(-m / Q)) / m))
    return Q, out

Q_b9, rows_b9 = overdamped_scan(rho_b9)
print(f"  rho = {rho_b9}:  Q = {Q_b9:.4f},  eps/m = {eps_over_m_b9:.4f},  "
      f"c_pred = 1/[2(eps/m)Q^2] = {1/(2*eps_over_m_b9*Q_b9**2):.2f}")
print(f"  {'eps':>6} {'m':>8} {'chi_sim':>10} {'chi_odrift':>12} {'rel.dev':>9}")
dev_b9 = []
for eps, m, chi, chi_qs in rows_b9:
    dev_b9.append(abs(chi_qs / chi - 1))
    print(f"  {eps:6.2f} {m:8.4f} {chi:10.4f} {chi_qs:12.4f} {dev_b9[-1]:9.2%}")
print(f"  slope at eps = {rows_b9[0][0]}: {(rows_b9[0][2] + 1 / Q_b9) / rows_b9[0][0]:.2f}"
      f"  (finite-eps value, bent by the higher powers of m/Q)")
report('B9 eq. (`eq:rest-od`) vs the full equations at Q=0.16',
       f'{max(dev_b9):.1%}', '<1.3%', max(dev_b9) < 0.013)

# slope as eps -> 0: Richardson extrapolation from eps = 1e-3 and 2e-3
_, rows_small = overdamped_scan(rho_b9, np.array([1.0e-3, 2.0e-3]))
sl = [(chi + 1 / Q_b9) / eps for eps, _, chi, _ in rows_small]
slope_b9 = 2 * sl[0] - sl[1]
c_full_b9 = 0.25 + (1 + Q_b9**-2) / (2 * eps_over_m_b9)     # eq. (restlaw)
c_twolimit = 0.25 + 1 / (2 * eps_over_m_b9 * Q_b9**2)       # Q << 1 limit only
print(f"  slopes at eps = 1e-3, 2e-3: {sl[0]:.4f}, {sl[1]:.4f};  "
      f"extrapolated to eps -> 0: c = {slope_b9:.4f}")
print(f"  eq. (restlaw) 1/4 + (m/2eps)(1 + Q^-2) = {c_full_b9:.4f};  "
      f"two-limit form 1/4 + 1/[2(eps/m)Q^2] = {c_twolimit:.2f} "
      f"({c_twolimit / slope_b9 - 1:+.1%})")
report('B9 slope c at Q=0.16 as eps->0 = 1/4 + (m/2eps)(1+Q^-2) = 20.4',
       f'{slope_b9:.4f}', f'{c_full_b9:.4f} +/- 0.1%',
       abs(slope_b9 / c_full_b9 - 1) < 1e-3)

# supporting check: the residual of eq. (`eq:rest-od`) is proportional to
# eps*Q, with coefficient 0.72, over a wide range of both.
# The quoted range is eps <= 0.04; beyond it the higher powers of eps that
# eq. (`eq:rest-od`) keeps and the linear law drops start to show.
print(f"  {'rho':>6} {'Q':>8} {'eps':>7} {'rel.dev':>9} {'dev/(eps Q)':>13}")
ratios = []
for rho_r in (3.0, 1.5, 0.75, 0.4):
    Q_r, rows_r = overdamped_scan(rho_r)
    for eps, m, chi, chi_qs in rows_r[:3]:
        r = abs(chi_qs / chi - 1) / (eps * Q_r)
        ratios.append(r)
        print(f"  {rho_r:6.2f} {Q_r:8.4f} {eps:7.3f} "
              f"{abs(chi_qs/chi - 1):9.2%} {r:13.3f}")
spread = max(ratios) / min(ratios) - 1
report('B9 (supporting) eq. (`eq:rest-od`) residual is 0.72 eps Q',
       f'{np.mean(ratios):.2f} +/- {100*spread:.0f}% over 7x in Q, 4x in eps',
       '0.72, flat to <15%', abs(np.mean(ratios) - 0.72) < 0.03 and spread < 0.15)


  rho = 1.5:  Q = 0.1557,  eps/m = 1.0504,  c_pred = 1/[2(eps/m)Q^2] = 19.63
     eps        m    chi_sim   chi_odrift   rel.dev
    0.01   0.0095    -6.2221      -6.2291     0.11%
    0.02   0.0190    -6.0310      -6.0444     0.22%
    0.04   0.0381    -5.6718      -5.6966     0.44%
    0.08   0.0762    -5.0364      -5.0787     0.84%
    0.12   0.1142    -4.4959      -4.5502     1.21%
  slope at eps = 0.01: 19.93  (finite-eps value, bent by the higher powers of m/Q)
[PASS] B9 eq. (`eq:rest-od`) vs the full equations at Q=0.16: measured=1.2%  threshold=<1.3%  


  slopes at eps = 1e-3, 2e-3: 20.3112, 20.2685;  extrapolated to eps -> 0: c = 20.3540
  eq. (restlaw) 1/4 + (m/2eps)(1 + Q^-2) = 20.3541;  two-limit form 1/4 + 1/[2(eps/m)Q^2] = 19.88 (-2.3%)
[PASS] B9 slope c at Q=0.16 as eps->0 = 1/4 + (m/2eps)(1+Q^-2) = 20.4: measured=20.3540  threshold=20.3541 +/- 0.1%  
     rho        Q     eps   rel.dev   dev/(eps Q)


    3.00   0.0779   0.010     0.06%         0.711
    3.00   0.0779   0.020     0.11%         0.696
    3.00   0.0779   0.040     0.21%         0.665


    1.50   0.1557   0.010     0.11%         0.720
    1.50   0.1557   0.020     0.22%         0.713
    1.50   0.1557   0.040     0.44%         0.700


    0.75   0.3115   0.010     0.23%         0.725
    0.75   0.3115   0.020     0.45%         0.724
    0.75   0.3115   0.040     0.90%         0.721
    0.40   0.5840   0.010     0.43%         0.729
    0.40   0.5840   0.020     0.85%         0.732
    0.40   0.5840   0.040     1.72%         0.738
[PASS] B9 (supporting) eq. (`eq:rest-od`) residual is 0.72 eps Q: measured=0.71 +/- 11% over 7x in Q, 4x in eps  threshold=0.72, flat to <15%  


### B10. Where the shift of the resting position comes from

> "This is exact. The bracket is the whole $\varepsilon$ effect: the
> rectified thrust and the change of $R$ along the path. The back-emf and
> the factor $(1+\varepsilon\chi)$ have dropped out ... Since all the launch
> energy is dissipated, $\int j_0^2ds=Q/2$ in every regime, and the thrust
> contributes exactly $\varepsilon/4$ ... $\int\chi_0j_0\,ds=Q/2+1/(2Q)$
> ... With $m/\varepsilon=0.952$, $c\simeq0.73$ above the critical point,
> whereas at $Q=0.16$ $c=20.4$" (body, Sec. 3.4)

Eq. (`findisp`) splits the shift of the resting position into a thrust
term and an $R$-variation term, and says the other two $\varepsilon$ terms
of eq. (`rescaled`) -- the back-emf $-\varepsilon\chi'j$ and the factor
$(1+\varepsilon\chi)$ -- cannot move it at first order. That is a strong
claim and an easy one to test directly: integrate eq. (`rescaled`) with
each term switched on or off and read off $c=(\chi_\infty+1/Q)/\varepsilon$
at small $\varepsilon$.

Three things are checked. (1) With the thrust alone, $c=1/4$ exactly, at
every $Q$. (2) With the $R$ variation alone, $c$ matches the integral
$(m/\varepsilon)Q^{-1}\int\chi_0j_0\,ds$ of eq. (`restlaw`), and that integral,
taken numerically on the $\varepsilon=0$ solution, equals the closed form
$Q/2+1/(2Q)$, so that $c=1/4+(m/2\varepsilon)(1+Q^{-2})$. (3) The two together reproduce
the full $c$, so the back-emf and $L(x)$ terms contribute nothing at this
order -- even though, switched off one at a time, each of them changes $c$
substantially.

In [19]:
m_over_eps = L_x0 / (Lp_x0 * (l + x0))          # = 1/(eps/m), eq. (LRexp)


def chi_inf_rescaled(eps, Q, thrust=1, Rvar=1, Lvar=1, backemf=1):
    """Eqs. (`chieq`)-(`rescaled`) with each eps term switchable, integrated
    to rest; returns chi at the end of the motion."""
    m = m_over_eps * eps

    def rhs(_, y):
        chi, chip, j = y
        jp = (chip * (1 - eps * j * backemf)
              - (1 + m * chi * Rvar) * j / Q) / (1 + eps * chi * Lvar)
        return [chip, -j + 0.5 * eps * j**2 * thrust, jp]

    sol = solve_ivp(rhs, [0.0, 200 * max(Q, 1 / Q)], [0.0, -1.0, 0.0],
                    rtol=1e-12, atol=1e-14)
    return sol.y[0, -1]


def c_integral(Q):
    """c of eq. (`restlaw`): 1/4 + (m/eps) (1/Q) int chi_0 j_0 ds, with the
    integral taken on the eps=0 solution."""
    def rhs(_, y):
        return [y[1], -y[2], y[1] - y[2] / Q, y[0] * y[2]]
    sol = solve_ivp(rhs, [0.0, 200 * max(Q, 1 / Q)], [0.0, -1.0, 0.0, 0.0],
                    rtol=1e-12, atol=1e-14)
    return 0.25 + m_over_eps * sol.y[3, -1] / Q


eps_b10 = 0.01
print(f"  m/eps = L(x0)/[L'(l+x0)] = {m_over_eps:.4f}")
print(f"  {'Q':>9} {'thrust only':>12} {'R var only':>11} {'both':>8} "
      f"{'all terms':>10} {'eq.(restlaw)':>10} {'closed':>9}")
worst_thrust, worst_sum, worst_cdef, worst_closed = 0.0, 0.0, 0.0, 0.0
for rho_r in (1.5, 0.75, 0.02, 0.01):
    Q_r = models.quality_factor(l, x0, B0, rho_r)
    c_of = lambda **kw: (chi_inf_rescaled(eps_b10, Q_r, **kw) + 1 / Q_r) / eps_b10
    c_thr = c_of(Rvar=0, Lvar=0, backemf=0)
    c_R = c_of(thrust=0, Lvar=0, backemf=0)
    c_both = c_of(Lvar=0, backemf=0)
    c_all = c_of()
    c_cd = c_integral(Q_r)
    lim = 0.25 + 0.5 * m_over_eps * (1 + Q_r**-2)   # closed form of eq. (restlaw)
    worst_thrust = max(worst_thrust, abs(c_thr - 0.25))
    worst_sum = max(worst_sum, abs(c_both / c_all - 1))
    worst_cdef = max(worst_cdef, abs(c_cd / c_all - 1))
    worst_closed = max(worst_closed, abs(c_cd / lim - 1))
    print(f"  {Q_r:9.4f} {c_thr:12.4f} {c_R:11.4f} {c_both:8.4f} "
          f"{c_all:10.4f} {c_cd:10.4f} {lim:9.4f}")

report('B10 thrust alone gives c = 1/4 at every Q', f'{worst_thrust:.1e}',
       '<1e-3', worst_thrust < 1e-3)
report('B10 thrust + R variation reproduce the full c', f'{worst_sum:.1%}',
       '<1%', worst_sum < 0.01)
report('B10 int chi_0 j_0 ds = Q/2 + 1/(2Q) on the eps=0 solution',
       f'{worst_closed:.1e}', '<1e-6', worst_closed < 1e-6)
report('B10 eq. (`restlaw`) predicts c from the eps=0 solution',
       f'{worst_cdef:.1%}', '<3%', worst_cdef < 0.03)

# switched off one at a time, each of the two "inert" terms matters a lot
Q_u = models.quality_factor(l, x0, B0, 0.02)
c_u = lambda **kw: (chi_inf_rescaled(eps_b10, Q_u, **kw) + 1 / Q_u) / eps_b10
print(f"  at Q = {Q_u:.2f}: c = {c_u():.4f} with every term, "
      f"{c_u(Lvar=0):.4f} without L(x), {c_u(backemf=0):.4f} without the "
      f"back-emf -- individually large, jointly inert.")


  m/eps = L(x0)/[L'(l+x0)] = 0.9520
          Q  thrust only  R var only     both  all terms eq.(restlaw)    closed


     0.1557       0.2499     19.7017  19.9417    19.9313    20.3541   20.3541
     0.3115       0.2498      5.3304   5.5754     5.5696     5.6330    5.6330


    11.6797       0.2499      0.4804   0.7302     0.7297     0.7295    0.7295


    23.3594       0.2499      0.4773   0.7272     0.7270     0.7269    0.7269
[PASS] B10 thrust alone gives c = 1/4 at every Q: measured=2.5e-04  threshold=<1e-3  
[PASS] B10 thrust + R variation reproduce the full c: measured=0.1%  threshold=<1%  
[PASS] B10 int chi_0 j_0 ds = Q/2 + 1/(2Q) on the eps=0 solution: measured=4.1e-12  threshold=<1e-6  
[PASS] B10 eq. (`restlaw`) predicts c from the eps=0 solution: measured=2.1%  threshold=<3%  


  at Q = 11.68: c = 0.7297 with every term, 1.2305 without L(x), 0.2295 without the back-emf -- individually large, jointly inert.


### B11. The textbook solution against the oscillator

> "the largest departure of (1) from the oscillator velocity (15), as a
> fraction of $v_0$, is below $1\%$ for $Q<0.1$ and reaches $14\%$ at the
> critical field itself, although the stopping distance $v_0/\alpha$ is the
> same in both" (body, Sec. 3.2.2)

Compare the textbook exponential `models.p_pos0` with the damped
oscillator `models.damped_oscillator` (both with $L$ and $R$ at $x_0$,
$M=1$) over the whole glide, at fields set to a chosen $Q=B_0/(2B_{0,c})$.
At critical damping the maximum is analytic: with $u=\gamma t$ and
$\alpha=\gamma/2$, $\max_u|e^{-u}(1+u)-e^{-u/2}|=0.1395$.

In [20]:
l, x0, rho_b11, p0_b11 = 100.0, 500.0, 1.0, -1.e-2
B0c_b11 = models.B0_critical(l, x0, rho_b11)


def textbook_error(Q):
    """max_t |v_osc - v_text| / |v0| and the two stopping distances."""
    B = 2 * Q * B0c_b11
    g = models.gamma_exact(l, x0, rho_b11)
    alpha = models.alpha0(B, rho_b11, l, x0)
    t = np.linspace(0, 40 / alpha, 400001)
    x_o, p_o, _ = models.damped_oscillator(t, x0, p0_b11, B, rho_b11, l)
    p_t = models.p_pos0(t, p0_b11, B, rho_b11, l, x0)
    return np.max(np.abs(p_o - p_t)) / abs(p0_b11), x_o[-1] - x0, p0_b11 / alpha


for Q in (0.02, 0.05, 0.1):
    e, *_ = textbook_error(Q)
    print(f"  Q = {Q:5.3f}: max |v_osc - v_text|/|v0| = {e:.3%}")
e_01, *_ = textbook_error(0.1)
e_c, dx_osc, dx_text = textbook_error(0.5 * (1 - 1e-9))
u = np.linspace(0, 20, 2000001)
e_an = np.max(np.abs(np.exp(-u) * (1 + u) - np.exp(-u / 2)))
print(f"  critical (Q = 1/2): {e_c:.4%}  (analytic {e_an:.4%});  "
      f"stopping distances: oscillator {dx_osc:.6f}, textbook {dx_text:.6f}")
report('B11 textbook error below 1% for Q<0.1', f'{e_01:.3%} at Q=0.1', '<1%', e_01 < 0.01)
report('B11 textbook error 14% at the critical field', f'{e_c:.2%}', '14% (rounded)',
       round(100 * e_c) == 14 and abs(e_c - e_an) < 1e-4)
report('B11 same stopping distance v0/alpha', f'{abs(dx_osc/dx_text - 1):.1e}', '<1e-6',
       abs(dx_osc / dx_text - 1) < 1e-6)

  Q = 0.020: max |v_osc - v_text|/|v0| = 0.040%
  Q = 0.050: max |v_osc - v_text|/|v0| = 0.244%
  Q = 0.100: max |v_osc - v_text|/|v0| = 0.931%
  critical (Q = 1/2): 13.9542%  (analytic 13.9542%);  stopping distances: oscillator -0.092634, textbook -0.092634
[PASS] B11 textbook error below 1% for Q<0.1: measured=0.931% at Q=0.1  threshold=<1%  
[PASS] B11 textbook error 14% at the critical field: measured=13.95%  threshold=14% (rounded)  
[PASS] B11 same stopping distance v0/alpha: measured=1.2e-13  threshold=<1e-6  


### B12. The laboratory-scale estimates and the reduced units

> "At laboratory scale ($M=10\,$g, $d=1\,$mm, $l=10\,$cm, $x_0=50\,$cm,
> copper) the critical field is $B_{0,c}\simeq3\,$T, and an oscillation as
> clear as that of Fig. 2 (right, $Q\simeq2.3$) needs $\sim14\,$T ...
> Scaling every length up by $100$ at $M=10\,$kg lowers these to
> $B_{0,c}\simeq1\,$mT and $B_0\simeq4.4\,$mT, with
> $\omega_0\simeq1.3\,$s$^{-1}$" (body, Sec. 4.1);
> "$\tau=7.5\times10^{-5}\,$s ... a moderate field ($B_0=2\times10^{-2}$,
> $\approx1\,$T)" (body, Secs. 2.4 and 3.3)

SI evaluation of (18) with the exact closed-form $L(x_0)$ (the closed form
is valid in any consistent units) and $R=2(l+x_0)\rho/(\pi d^2)$,
$\rho_{Cu}=1.68\times10^{-8}\,\Omega\,$m. The reduced unit of field
follows from those of mass, length and time: force balance
$M\dot v=-lB_0I$ and the magnetic energy $\frac12LI^2$, with
$L\propto\mu_0d$, give a current unit $\sqrt{Md/\mu_0}/\tau$ and a field
unit $\sqrt{\mu_0M/d}/\tau$, about $47\,$T for the bench parameters.

In [21]:
MU0, RHO_CU = 4e-7 * np.pi, 1.68e-8


def bench_numbers(M, d, l_si, x_si, Q=2.3):
    L = inductance.self_inductance(MU0, d, l_si, x_si)
    R = 2 * (l_si + x_si) * RHO_CU / (np.pi * d**2)
    g = R / (2 * L)
    Bc = np.sqrt(M * L) * g / l_si
    B = 2 * Q * Bc                       # Q = B/(2 Bc)
    w0 = l_si * B / np.sqrt(M * L)
    return Bc, B, w0


Bc1, B1, _ = bench_numbers(0.01, 1e-3, 0.1, 0.5)
Bc2, B2, w02 = bench_numbers(10.0, 0.1, 10.0, 50.0)
tau = MU0 * (1e-3)**2 / RHO_CU
B_unit = np.sqrt(MU0 * 0.01 / 1e-3) / tau
print(f"  bench:     B0c = {Bc1:.3f} T,  B0(Q=2.3) = {B1:.2f} T")
print(f"  x100:      B0c = {Bc2*1e3:.3f} mT, B0(Q=2.3) = {B2*1e3:.2f} mT, omega0 = {w02:.3f} 1/s")
print(f"  tau = {tau:.3e} s,  field unit = {B_unit:.2f} T,  B0 = 2e-2 -> {0.02*B_unit:.3f} T")
report('B12 bench B0c ~ 3 T', f'{Bc1:.2f} T', '3 T +/- 5%', abs(Bc1 / 3 - 1) < 0.05)
report('B12 bench B0 for Q=2.3 ~ 14 T', f'{B1:.1f} T', '14 T +/- 5%', abs(B1 / 14 - 1) < 0.05)
report('B12 x100 B0c ~ 1 mT, B0 ~ 4.4 mT, omega0 ~ 1.3 1/s',
       f'{Bc2*1e3:.2f} mT, {B2*1e3:.2f} mT, {w02:.2f} 1/s', 'each +/- 5%',
       abs(Bc2 / 1e-3 - 1) < 0.05 and abs(B2 / 4.4e-3 - 1) < 0.05 and abs(w02 / 1.3 - 1) < 0.05)
report('B12 tau = 7.5e-5 s and B0 = 2e-2 ~ 1 T', f'{tau:.2e} s, {0.02*B_unit:.2f} T',
       '7.5e-5 s, 1 T +/- 10%', abs(tau / 7.5e-5 - 1) < 0.01 and abs(0.02 * B_unit - 1) < 0.1)

  bench:     B0c = 3.043 T,  B0(Q=2.3) = 14.00 T
  x100:      B0c = 0.962 mT, B0(Q=2.3) = 4.43 mT, omega0 = 1.328 1/s
  tau = 7.480e-05 s,  field unit = 47.39 T,  B0 = 2e-2 -> 0.948 T
[PASS] B12 bench B0c ~ 3 T: measured=3.04 T  threshold=3 T +/- 5%  
[PASS] B12 bench B0 for Q=2.3 ~ 14 T: measured=14.0 T  threshold=14 T +/- 5%  
[PASS] B12 x100 B0c ~ 1 mT, B0 ~ 4.4 mT, omega0 ~ 1.3 1/s: measured=0.96 mT, 4.43 mT, 1.33 1/s  threshold=each +/- 5%  
[PASS] B12 tau = 7.5e-5 s and B0 = 2e-2 ~ 1 T: measured=7.48e-05 s, 0.95 T  threshold=7.5e-5 s, 1 T +/- 10%  


### B13. The two runs of Fig. 2

> "On the left, a moderate field ... gives $Q=0.16$ ... On the right, a
> field fifteen times larger gives $Q=2.3$ ... the largest departure is
> $8\times10^{-4}$ of the full range of $x$ on the left and $2\times10^{-4}$
> on the right, small because the $\varepsilon$ of the two runs,
> $2.6\times10^{-4}$ and $1.7\times10^{-5}$, is small" (body, Sec. 3.3);
> "The runs of Fig. 2 have $\varepsilon<3\times10^{-4}$" (body, Sec. 3.4)

Repeats both runs at the paper's step counts ($\Delta t=2\times10^{-4}$,
$4\times10^6$ and $10^6$ steps), measures the largest $|x-x_{osc}|$ as a
fraction of the range of $x$, and evaluates
$\varepsilon=L'|v_0|/(\omega_0L(x_0))$ (21).

In [22]:
l, x0, rho_b13, p0_b13, dt_b13 = 100.0, 500.0, 1.0, -1.e-2, 2.e-4
L_b13 = inductance.self_inductance(1.0, 1.0, l, x0)
Lp_b13 = inductance.self_inductance_dx(1.0, 1.0, l, x0)
res_b13 = {}
for B, n_steps, n_print in [(0.02, 4_000_000, 400), (0.3, 1_000_000, 20)]:
    xs, *_ = dynamics.verlet_integrate(x0, p0_b13, l, B, rho_b13, dt_b13, n_steps, n_print)
    xs = np.array(xs)
    t = n_print * dt_b13 * np.arange(len(xs))
    x_o, _, _ = models.damped_oscillator(t, x0, p0_b13, B, rho_b13, l)
    dep = np.max(np.abs(xs - x_o)) / (xs.max() - xs.min())
    eps = Lp_b13 * abs(p0_b13) / (models.omega0(l, x0, B) * L_b13)
    Q = models.quality_factor(l, x0, B, rho_b13)
    res_b13[B] = (Q, eps, dep)
    print(f"  B0 = {B}: Q = {Q:.4f}, eps = {eps:.3e}, max|x - x_osc|/range = {dep:.2e}")
(Q_a, e_a, d_a), (Q_b, e_b, d_b) = res_b13[0.02], res_b13[0.3]


def sig1(v):
    return float(f'{v:.0e}')


report('B13 Q = 0.16 and 2.3', f'{Q_a:.3f}, {Q_b:.3f}', 'round to 0.16, 2.3',
       round(Q_a, 2) == 0.16 and round(Q_b, 1) == 2.3)
report('B13 eps = 2.6e-4 and 1.7e-5 (< 3e-4)', f'{e_a:.2e}, {e_b:.2e}', '2.6e-4, 1.7e-5',
       round(e_a, 5) == 2.6e-4 and round(e_b, 6) == 1.7e-5 and max(e_a, e_b) < 3e-4)
report('B13 largest departures 8e-4 and 2e-4 of the range of x', f'{d_a:.2e}, {d_b:.2e}',
       '8e-4, 2e-4 (1 significant figure)', sig1(d_a) == 8e-4 and sig1(d_b) == 2e-4)

  B0 = 0.02: Q = 0.1557, eps = 2.603e-04, max|x - x_osc|/range = 8.23e-04


  B0 = 0.3: Q = 2.3359, eps = 1.736e-05, max|x - x_osc|/range = 1.83e-04
[PASS] B13 Q = 0.16 and 2.3: measured=0.156, 2.336  threshold=round to 0.16, 2.3  
[PASS] B13 eps = 2.6e-4 and 1.7e-5 (< 3e-4): measured=2.60e-04, 1.74e-05  threshold=2.6e-4, 1.7e-5  
[PASS] B13 largest departures 8e-4 and 2e-4 of the range of x: measured=8.23e-04, 1.83e-04  threshold=8e-4, 2e-4 (1 significant figure)  


### B14. The frequency signature of Sec. 3.4

> "the peak-to-peak spread of the instantaneous frequency is
> $0.98\,\varepsilon$ at $Q=11.7$ ($39\%$ at $\varepsilon=0.40$), and the
> ring-down ends not at $\omega_d(x_0)=0.1008$ but at $0.0968$, the value
> at the bar's *final* position" (body, Sec. 3.4); "with a coefficient that
> grows towards $\approx1.27$ as the damping is weakened and falls below $1$
> at stronger damping ($0.98$ at $Q=11.7$)" (supplementary derivations,
> "Frequency modulation")

Ring-downs at $B_0=0.03$ from $|p_0|=\varepsilon lB_0\sqrt{L(x_0)}/L'$,
$\Delta t=10^{-2}$, with the frequency read off consecutive zero crossings
of $I(t)$ (`models.instantaneous_frequency`).

In [23]:
l, x0, B0_s8, dt_s8 = 100.0, 500.0, 0.03, 1.0e-2


def ring_down_s8(eps, rho, n_tau=20, n_print=10, dt=dt_s8):
    g = models.gamma_exact(l, x0, rho)
    p0 = -models.p0_from_epsilon(eps, l, x0, B0_s8)
    xs, ps, Is, *_ = dynamics.verlet_integrate(
        x0, p0, l, B0_s8, rho, dt, int(n_tau / g / dt), n_print)
    return n_print * dt * np.arange(len(xs)), np.array(xs), np.array(Is)


def spread(t, I):
    _, w = models.instantaneous_frequency(t, I)
    return (w.max() - w.min()) / w.mean(), w


coef = []
for eps in (0.02, 0.10, 0.20, 0.40):
    t, xs, Is = ring_down_s8(eps, 0.02)
    sp, w = spread(t, Is)
    coef.append(sp / eps)
    print(f"  Q = 11.7, eps = {eps:.2f}: spread = {sp:.4f}, spread/eps = {sp/eps:.4f}, "
          f"final omega = {w[-1]:.5f}")
sp_04, w_04 = sp, w
g_s8 = models.gamma_exact(l, x0, 0.02)
wd_x0 = np.sqrt(models.omega0(l, x0, B0_s8)**2 - g_s8**2)
wd_end = np.sqrt(models.omega0(l, xs[-1], B0_s8)**2 - g_s8**2)
print(f"  omega_d(x0) = {wd_x0:.5f},  omega_d(x_inf) = {wd_end:.5f} at x_inf = {xs[-1]:.2f}")
report('B14 spread/eps = 0.98 at Q=11.7', f'{np.mean(coef):.3f}', '0.98 (2 d.p.)',
       round(np.mean(coef), 2) == 0.98)
report('B14 spread 39% at eps=0.40', f'{sp_04:.3f}', '0.39', round(sp_04, 2) == 0.39)
report('B14 omega_d(x0)=0.1008, final omega=0.0968 = omega_d(x_inf)',
       f'{wd_x0:.5f}, {w_04[-1]:.5f}, {wd_end:.5f}', '0.1008, 0.0968, equal to 1e-4',
       round(wd_x0, 4) == 0.1008 and round(w_04[-1], 4) == 0.0968 and abs(w_04[-1] - wd_end) < 1e-4)

coef_Q = []
for rho in (0.02, 0.005, 0.001):
    t, xs, Is = ring_down_s8(0.02, rho, n_tau=4, n_print=5)
    sp, _ = spread(t, Is)
    coef_Q.append(sp / 0.02)
    print(f"  eps = 0.02, Q = {models.quality_factor(l, x0, B0_s8, rho):6.1f}: spread/eps = {sp/0.02:.4f}")
report('B14 coefficient grows towards ~1.27 as the damping weakens',
       ', '.join(f'{c:.3f}' for c in coef_Q), 'increasing, last within 2% of 1.27',
       all(np.diff(coef_Q) > 0) and abs(coef_Q[-1] / 1.27 - 1) < 0.02)

  Q = 11.7, eps = 0.02: spread = 0.0195, spread/eps = 0.9749, final omega = 0.10085


  Q = 11.7, eps = 0.10: spread = 0.0976, spread/eps = 0.9761, final omega = 0.10084


  Q = 11.7, eps = 0.20: spread = 0.1955, spread/eps = 0.9774, final omega = 0.10017


  Q = 11.7, eps = 0.40: spread = 0.3912, spread/eps = 0.9779, final omega = 0.09682
  omega_d(x0) = 0.10077,  omega_d(x_inf) = 0.09682 at x_inf = 547.48
[PASS] B14 spread/eps = 0.98 at Q=11.7: measured=0.977  threshold=0.98 (2 d.p.)  
[PASS] B14 spread 39% at eps=0.40: measured=0.391  threshold=0.39  
[PASS] B14 omega_d(x0)=0.1008, final omega=0.0968 = omega_d(x_inf): measured=0.10077, 0.09682, 0.09682  threshold=0.1008, 0.0968, equal to 1e-4  
  eps = 0.02, Q =   11.7: spread/eps = 0.9750


  eps = 0.02, Q =   46.7: spread/eps = 1.1906


  eps = 0.02, Q =  233.6: spread/eps = 1.2564
[PASS] B14 coefficient grows towards ~1.27 as the damping weakens: measured=0.975, 1.191, 1.256  threshold=increasing, last within 2% of 1.27  


### B15. The runs of Sec. 3.4: constants, range, the linear law and the reversal

> "the runs use the exact $L(x_0)$ and $dL/dx$ at $x_0$, both within about $0.2\%$
> of (2) ... Over all the runs of this subsection, including the
> $\varepsilon$ sweeps behind the values of $c$ and $\varepsilon^*$ quoted
> below, the bar stays between $x\simeq270$ and $x\simeq850$, where (2) is
> accurate ... With $m/\varepsilon=0.952$ ... $\varepsilon^*\simeq1/(cQ)$,
> measured at $0.117$ for $Q=11.7$ ... at $\varepsilon=0.40$, it ends $9.5\%$
> of $x_0$ beyond $x_0$, where the oscillator predicts $3.9\%$ short of it"
> (body, Sec. 3.4); "$L_0\simeq113.5\,\mu_0d$ ... $L(x_0=500d)\simeq884.6\,\mu_0d$"
> (body, Sec. 2.2); "Integrating the full equations of motion for
> $0.02\le\varepsilon\le0.5$ and fitting $\chi_\infty-\sigma/Q$ against
> $\varepsilon$ returns $c=0.733$, $0.726$ and $0.724$ at $Q=11.7$, $23.4$
> and $46.7$ ... The measured thresholds ... are $\varepsilon^*=0.117$,
> $0.059$ and $0.030$" (supplementary derivations, "The resting position of
> the bar")

The sweep is the one of Notebook 1 at full resolution (nine values of
$\varepsilon$, three resistivities, each run for $20/\gamma$). The range of
$x$ is taken over the four runs of Fig. 3 and the first three periods of
every run of the sweep, where the extreme excursions occur.

In [24]:
L_s8 = inductance.self_inductance(1.0, 1.0, l, x0)
Lp_s8 = inductance.self_inductance_dx(1.0, 1.0, l, x0)
Lp_as = inductance.self_inductance_dx_asymp(1.0, 1.0, l)
L_as = inductance.self_inductance_asymp(1.0, 1.0, l, x0)
L0_as = L_as - Lp_as * x0
m_over_eps_s8 = L_s8 / (Lp_s8 * (x0 + l))
print(f"  L(x0) = {L_s8:.2f} (asymptotic {L_as:.2f}),  dL/dx(x0) = {Lp_s8:.5f} "
      f"(L' = {Lp_as:.5f}),  L0 = {L0_as:.2f} = {L0_as/(l*Lp_as):.3f} l L',  m/eps = {m_over_eps_s8:.4f}")
report('B15 L0 = 113.5 = 0.73 l L\', L(x0) = 884.6', f'{L0_as:.1f}, {L0_as/(l*Lp_as):.2f}, {L_s8:.1f}',
       '113.5, 0.73, 884.6', round(L0_as, 1) == 113.5 and round(L0_as / (l * Lp_as), 2) == 0.73
       and round(L_s8, 1) == 884.6)
report('B15 exact L(x0) and dL/dx(x0) within about 0.2% of eq. (2)',
       f'{abs(L_s8/L_as-1):.3%}, {abs(Lp_s8/Lp_as-1):.3%}', '0.2% (1 d.p.) each',
       round(100 * abs(L_s8 / L_as - 1), 1) <= 0.2 and round(100 * abs(Lp_s8 / Lp_as - 1), 1) <= 0.2)
report('B15 m/eps = 0.952', f'{m_over_eps_s8:.4f}', '0.952', round(m_over_eps_s8, 3) == 0.952)

# --- range of x over the runs of Fig. 3 and the sweep
w0_s8 = models.omega0(l, x0, B0_s8)
T0_s8 = 2 * np.pi / w0_s8
lo, hi = np.inf, -np.inf
for rho, eps, n_per in [(1.5, 0.002, 5), (1.5, 0.08, 5), (0.02, 0.002, 20), (0.02, 0.40, 20)]:
    p0 = -models.p0_from_epsilon(eps, l, x0, B0_s8)
    xs, *_ = dynamics.verlet_integrate(x0, p0, l, B0_s8, rho, dt_s8, int(n_per * T0_s8 / dt_s8), 10)
    lo, hi = min(lo, min(xs)), max(hi, max(xs))
eps_sweep = np.array([0.02, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50])
for rho in (0.02, 0.01, 0.005):
    for eps in eps_sweep:
        p0 = -models.p0_from_epsilon(eps, l, x0, B0_s8)
        xs, *_ = dynamics.verlet_integrate(x0, p0, l, B0_s8, rho, dt_s8, int(3 * T0_s8 / dt_s8), 10)
        lo, hi = min(lo, min(xs)), max(hi, max(xs))
print(f"  range of x over the runs: {lo:.2f} to {hi:.2f}")
report('B15 the bar stays between x ~ 270 and x ~ 850', f'{lo:.1f} to {hi:.1f}',
       '270 and 850 to the nearest 10', round(lo, -1) == 270 and round(hi, -1) == 850)

# --- the linear law and the reversal, as in Notebook 1
c_meas, es_meas = {}, {}
for rho in (0.02, 0.01, 0.005):
    Q = models.quality_factor(l, x0, B0_s8, rho)
    chi = []
    for eps in eps_sweep:
        t, xs, _ = ring_down_s8(eps, rho)
        chi.append((xs[-1] - x0) / models.free_amplitude(eps, l, x0))
        if rho == 0.02 and eps == 0.40:
            dx_04 = (xs[-1] - x0) / x0
    c, es = models.fit_drift_linear(eps_sweep, np.array(chi), Q)
    c_meas[rho], es_meas[rho] = c, es
    print(f"  rho = {rho}: Q = {Q:.2f}, c = {c:.4f}, eps* = {es:.4f}")
Q_117 = models.quality_factor(l, x0, B0_s8, 0.02)
dx_osc = -(L_s8 / Lp_s8) * 0.40 / Q_117 / x0
print(f"  eps = 0.40, Q = 11.7: dx/x0 = {dx_04:+.4f}, oscillator {dx_osc:+.4f}")
report('B15 c = 0.733, 0.726, 0.724', ', '.join(f'{c_meas[r]:.3f}' for r in (0.02, 0.01, 0.005)),
       '0.733, 0.726, 0.724', [round(c_meas[r], 3) for r in (0.02, 0.01, 0.005)] == [0.733, 0.726, 0.724])
report('B15 eps* = 0.117, 0.059, 0.030', ', '.join(f'{es_meas[r]:.4f}' for r in (0.02, 0.01, 0.005)),
       '0.117, 0.059, 0.030', [round(es_meas[r], 3) for r in (0.02, 0.01, 0.005)] == [0.117, 0.059, 0.030])
report('B15 at eps=0.40: +9.5% of x0, oscillator -3.9%', f'{dx_04:+.2%}, {dx_osc:+.2%}',
       '+9.5%, -3.9%', round(100 * dx_04, 1) == 9.5 and round(100 * dx_osc, 1) == -3.9)

  L(x0) = 884.59 (asymptotic 886.18),  dL/dx(x0) = 1.54862 (L' = 1.54545),  L0 = 113.46 = 0.734 l L',  m/eps = 0.9520
[PASS] B15 L0 = 113.5 = 0.73 l L', L(x0) = 884.6: measured=113.5, 0.73, 884.6  threshold=113.5, 0.73, 884.6  
[PASS] B15 exact L(x0) and dL/dx(x0) within about 0.2% of eq. (2): measured=0.179%, 0.205%  threshold=0.2% (1 d.p.) each  
[PASS] B15 m/eps = 0.952: measured=0.9520  threshold=0.952  


  range of x over the runs: 269.93 to 849.32
[PASS] B15 the bar stays between x ~ 270 and x ~ 850: measured=269.9 to 849.3  threshold=270 and 850 to the nearest 10  


  rho = 0.02: Q = 11.68, c = 0.7332, eps* = 0.1168


  rho = 0.01: Q = 23.36, c = 0.7263, eps* = 0.0589


  rho = 0.005: Q = 46.72, c = 0.7235, eps* = 0.0296
  eps = 0.40, Q = 11.7: dx/x0 = +0.0950, oscillator -0.0391
[PASS] B15 c = 0.733, 0.726, 0.724: measured=0.733, 0.726, 0.724  threshold=0.733, 0.726, 0.724  
[PASS] B15 eps* = 0.117, 0.059, 0.030: measured=0.1168, 0.0589, 0.0296  threshold=0.117, 0.059, 0.030  
[PASS] B15 at eps=0.40: +9.5% of x0, oscillator -3.9%: measured=+9.50%, -3.91%  threshold=+9.5%, -3.9%  


### B16. The step size of the Sec. 3.4 runs

> "a step $\Delta t=10^{-2}$ suffices for them: quartering it changes the
> results by about $10^{-7}$ relative." (supplementary derivations,
> "Velocity-Verlet integration scheme", "Step size")

The largest-$\varepsilon$ run on each side of the critical point, at
$\Delta t=10^{-2}$ and $2.5\times10^{-3}$; the measured quantity is the
displacement $x_\infty-x_0$.

In [25]:
rel = []
for rho, eps in [(0.02, 0.40), (1.5, 0.08)]:
    ends = []
    for dt in (1.0e-2, 2.5e-3):
        if rho < 0.5:
            t, xs, _ = ring_down_s8(eps, rho, n_print=100, dt=dt)
        else:
            p0 = -models.p0_from_epsilon(eps, l, x0, B0_s8)
            xs, *_ = dynamics.verlet_integrate(x0, p0, l, B0_s8, rho, dt,
                                               int(40 * T0_s8 / dt), 100)
        ends.append(xs[-1])
    rel.append(abs(ends[0] - ends[1]) / abs(ends[1] - x0))
    print(f"  rho = {rho}, eps = {eps}: x_inf = {ends[0]:.8f} (dt=1e-2), "
          f"{ends[1]:.8f} (dt=2.5e-3), rel. change {rel[-1]:.1e}")
report('B16 quartering dt=1e-2 changes x_inf - x0 by ~1e-7', f'{max(rel):.1e}', '<1e-6',
       max(rel) < 1e-6)

  rho = 0.02, eps = 0.4: x_inf = 547.48330019 (dt=1e-2), 547.48329553 (dt=2.5e-3), rel. change 9.8e-08


  rho = 1.5, eps = 0.08: x_inf = 269.84927072 (dt=1e-2), 269.84927063 (dt=2.5e-3), rel. change 4.2e-10
[PASS] B16 quartering dt=1e-2 changes x_inf - x0 by ~1e-7: measured=9.8e-08  threshold=<1e-6  


## Summary

In [26]:
print(f"{len(PASS)} passed, {len(FAIL)} failed, out of {len(PASS)+len(FAIL)} checks.")
if FAIL:
    print("FAILED:")
    for c in FAIL:
        print(" -", c)
else:
    print("Every claim reproduced here matches the paper's stated threshold.")


59 passed, 0 failed, out of 59 checks.
Every claim reproduced here matches the paper's stated threshold.
